  <img src="../images/mackenzie_logo.jpg" width="25%" align="right"/>

# **PROJETO APLICADO IV - Ciência de Dados EaD - 2026/02**

# **Documentação do Projeto**

---

**Versão:** Etapa 4, entrega em 30/11/2026. O texto das Etapas 1, 2 e 3 permanece. O incremento acrescenta a avaliação corrigida de um passo, a grade financeira trimestral e a discussão dos resultados contemporâneos. A base é o export de 16/10/2026, corte em 21/09/2026 (`data/gas_flare.sqlite`, SHA-256 `b314283c8e711becd802d5162be38b1c47052e9603d8bbf81345916fc2bf35ba`). O ensaio PETR4 − BOVA11 lê o suplemento `data/etapa4_bova11_20261018T152955Z.sqlite` e o overlay `data/etapa4_equity_overlay_2023-11-17.json`. A grade ARIMA/SARIMA da Etapa 4 reproduz no WSL, kernel `gas` (Python 3.12.13, statsmodels 0.14.6, numpy 2.4.3, pandas 2.3.3, scikit-learn 1.8.0). Este caderno foi executado em kernel limpo nesse ambiente em 18/10/2026.


# **Paridade de preços de combustíveis, atividade econômica e a ação da Petrobras**
---


In [1]:
#@title **Identificação do Grupo**

#@markdown Integrantes do Grupo, nome completo em ordem alfabética (*informe: \<nome\>, \<matrícula\>*)
#@markdown AVA: grupo 19
Aluno1 = 'Guilherme Augusto Leal Oliveira, 10226108' #@param {type:"string"}
Aluno2 = 'Guilherme Rocha de Souza Duarte, 10730632' #@param {type:"string"}
Aluno3 = 'Ricardo Zulian de Souza Amaral, 10433651' #@param {type:"string"}
Aluno4 = '' #@param {type:"string"}
Aluno5 = '' #@param {type:"string"}


# **Resumo**

O grupo analisou se o desvio de paridade de combustíveis ponderado pelo volume acompanha preços, atividade econômica, PETR4 (ação preferencial da Petrobras na B3) e as contas de RT&M (Refino, Transporte e Comercialização) da Petrobras. O estudo usa o arquivo SQLite congelado, os modelos de referência da Etapa 3 e a regressão Ridge, uma regressão linear com penalização dos coeficientes, em avaliação temporal de um passo: em cada data, a estimativa usa valores anteriores observados do alvo e regressores contemporâneos realizados. Na comparação com combustíveis separados (`y_fuels`), a raiz do erro quadrático médio (RMSE) mediana do diesel da Agência Nacional do Petróleo, Gás Natural e Biocombustíveis (ANP) cai 16,7%, mas o RMSE é menor em apenas 1 de 4 blocos de datas; a exposição Petrobras contemporânea (`y_pbr0`) tem RMSE menor em 3 de 4 blocos, e as duas extensões não são o mesmo regressor. No Índice Nacional de Preços ao Consumidor Amplo (IPCA), a variação do preço ANP da gasolina reduz o RMSE mediano em 20,4% e o RMSE é menor em 4 de 4 blocos; esse regressor é a variação do preço ANP de produtores e importadores, não o desvio de paridade ponderado pelo volume.

Na grade financeira de 76 configurações, o grupo escolheu o método com os trimestres de 2019 a 2021 e manteve essa escolha na avaliação de 2022 a 2026 (a trava). A receita permanece com persistência; o lucro, com SARIMA(0,1,0)×(1,0,0)₄; a margem, com SARIMA(0,0,0)×(0,0,0)₄, um nível médio da série. SARIMA é a média móvel integrada auto-regressiva sazonal; neste caso da margem, os índices nulos indicam um nível constante, não um ciclo trimestral. Nenhum dos três reduz em 5% ou mais o RMSE agrupado, calculado sobre os trimestres juntos, frente à persistência (copiar o último trimestre) na avaliação do 1º trimestre de 2022 ao 1º trimestre de 2026. A correção do resíduo soma à previsão já escolhida um termo ligado à exposição de diesel: reduz o RMSE do lucro e da margem do 1º trimestre de 2024 ao 1º trimestre de 2026 e aumenta o erro das três séries quando o 1º trimestre de 2026 sai da amostra. O ensaio descreve a associação contemporânea e não promove um modelo de previsão. A extensão de retorno relativo da PETR4 frente ao BOVA11 não atendeu ao critério incremental nas duas dimensões (Tabelas 5 e 6). O notebook reconstrói os experimentos desde a base. A viabilidade ex ante, prever só com o que já teria sido publicado na data da origem, não foi executada.


# **Introdução**

## Contexto

O Brasil combina preço na refinaria com a paridade internacional como referência de mercado. O PPI (Índice de Paridade de Preços) da ABICOM publica, em dias úteis, o diferencial em R＄/L e em % entre o preço Petrobras e a paridade de importação para gasolina e diesel (ABICOM, 2026). O diesel entra no frete; a gasolina, no tráfego leve das cidades. A Petrobras é empresa de economia mista listada na B3; o papel PETR4 reage a reajuste e à paridade. O tema é série temporal com frequências mistas: dia, semana, mês e trimestre.

O recorte liga três Objetivos de Desenvolvimento Sustentável. O ODS 8 trata de trabalho e crescimento: diesel, tráfego pesado, indústria e desemprego. O ODS 11 trata de cidades: gasolina, tráfego leve e preço ao consumidor. O ODS 16 trata de instituições: a política de preços de uma empresa estatal de capital aberto e misto e o PPI como sinal público do desvio em relação à paridade e ao interesse do acionista.

## Motivação

A política de preços da Petrobras mudou mais de uma vez nesta década. O mercado observa o preço internacional dos derivados e o PPI, mas a transmissão até o posto, o volume e as variáveis socioeconômicas não é uniforme. Fontes públicas (ANP, IBGE, ABCR, B3, BCB) publicam calendários distintos. O grupo junta essas séries em um recorte único a partir de 2022 e lê cada alvo contra o produto desvio × volume (`prod`) do mesmo período.

## Objetivo geral

O objetivo do projeto é medir o efeito do desvio de paridade ponderado pelo volume (`prod`) sobre PETR4, o preço de combustível, o tráfego, a indústria, o varejo, o desemprego e o resultado trimestral da Petrobras, e usar esse produto como previsório contemporâneo.

Objetivos específicos:

1. Organizar a base congelada com âncora no PPI diário.
2. Medir defasagens entre PPI, preço ANP/Petrobras e PETR4.
3. Testar se choques de diesel antecipam tráfego pesado, indústria e desemprego (ODS 8).
4. Testar se choques de gasolina e tráfego leve descrevem mobilidade urbana (ODS 11).
5. Documentar o PPI como sinal público da política de preços da Petrobras (ODS 16).
6. Entregar um notebook reproduzível com modelo estatístico de série temporal.

A hipótese de trabalho é que o choque de paridade e de preço passa primeiro pela ação e, depois, pela indústria, pelo varejo, pelo emprego e pelo tráfego.

## Justificativa

Um dos integrantes do grupo criou o pipeline ETL que captura e armazena esses dados. O pipeline está ativo. Nenhum estudo foi produzido a partir deste pipeline.

A contribuição analítica é um recorte reproduzível de fontes originais, com frequência mista e âncora diária, sem base pré-processada de terceiro. A contribuição prática é a leitura do `prod` contra preço, frete, mobilidade, emprego e resultado da companhia.

O ODS 8 liga diesel, emprego e crescimento. O ODS 11 liga gasolina, tráfego leve e cidades. O ODS 16 liga a transparência da política de preços à Petrobras como instituição.

O produto do curso é um experimento executável. Não é um painel de produção.

## Descrição da base de dados

A base do curso é o arquivo `data/gas_flare.sqlite`. O arquivo é um snapshot congelado. O projeto não acessa Postgres, API ao vivo nem credencial. As séries vêm de fontes originais (governo e entidades setoriais), reunidas em 14 tabelas. A âncora temporal é o PPI diário: 5 de janeiro de 2022 a 21 de setembro de 2026, com 1.151 datas. As demais famílias entram nesse calendário de dias úteis, cada uma com a defasagem da fonte. O Quadro 1 apresenta as famílias da base congelada.

Quadro 1 — Famílias da base congelada

| Família | Frequência | Conteúdo | Recorte no snapshot | Fonte |
|---|---|---|---|---|
| PPI | diária | diferencial Petrobras e independentes vs. paridade, gasolina e diesel, R＄/L e % | 5 jan. 2022 a 21 set. 2026 (1.151 datas) | ABICOM |
| Ação PETR4 | diária (pregão) | abertura, máxima, mínima, fechamento, volume | 1º fev. 2022 a 18 set. 2026 (1.156 pregões) | B3 |
| Preço ANP | semanal | média ao produtor/importador, linha Brasil: gasolina A comum; diesel S-10 e S-500, R＄/L | 3 jan. 2022 a 13 set. 2026 (245 semanas por produto) | ANP |
| Preço Petrobras | semanal irregular | preço de lista às distribuidoras, sem tributos, gasolina A e diesel S-10 e S-500, R＄/L | jan. 2022 a set. 2026 (64 semanas de gasolina; 72 de diesel) | Petrobras |
| Vendas ANP | mensal | volume nacional, m³ | jan. 2022 a jul. 2026 | ANP |
| Vendas e finanças Petrobras | trimestral | volume (mbpd), receita e margem RT&M | vendas 2022T1–2026T2; finanças de 2006T1 a 2026T2 | Petrobras |
| Tráfego | mensal | índice ABCR leve, pesado e total | fev. 2022 a ago. 2026 | ABCR |
| Atividade e emprego | mensal | desemprego (PNAD), varejo (PMC), indústria (PIM-PF) | jan. 2022 a jul. 2026 | IBGE |
| Inflação | mensal | IPCA, IPCA-15 e IGP-M | jan. 2022 a ago. 2026 | IBGE / FGV via BCB |

A coleta operacional de origem é semanal. O curso usa o estado congelado. Semanas sem PDF novo da Petrobras não avançam a série de preço à distribuidora: isso é ausência de reajuste, não falha de carga. PETR4 começa em fevereiro de 2022; os NaNs de janeiro no índice do PPI são esperados. Os preços da B3 não têm ajuste por proventos. O dicionário das tabelas está em `docs/metadados_base.md`.

### Atualização da base

A revisão da Etapa 2 conferiu a base congelada contra as fontes e encontrou erros de origem. O grupo levou esses erros ao pipeline que alimenta a base, e o pipeline corrigiu os pontos abaixo:

- PPI: leituras de OCR erradas, como −13,0 R＄/L em 26 de janeiro de 2023 e percentuais de três dígitos sem o primeiro algarismo em setembro de 2026.
- Preço ANP: o diesel misturava biodiesel B-100 e diesel marítimo ao diesel rodoviário, e a gasolina incluía a gasolina Premium. A carga também somava 22 cópias da linha Brasil como se fossem regiões.
- Preço Petrobras: algumas semanas vinham de tabelas de outro produto.
- Finanças Petrobras: os trimestres 1T26 e 2T26 estavam vazios.

Depois das correções, o grupo exportou uma nova base, com corte em 21 de setembro de 2026. O Quadro 1 descreve essa base. A versão anterior fica em `data/gas_flare_old.sqlite`, e a tabela `export_changelog` registra cada correção com a causa e a maior alteração de valor.


# **Referencial Teórico**

## Conceitos

Uma série temporal é um conjunto de observações coletadas em intervalo regular (HYNDMAN; ATHANASOPOULOS, 2021). Neste projeto, o PPI e o pregão de PETR4 seguem os dias úteis, enquanto preço, volume, tráfego e atividade chegam por semana, mês ou trimestre; reunidas no mesmo calendário, essas séries formam uma série multivariada.

A leitura de cada série separa quatro componentes: a tendência, que indica se a série sobe, fica estável ou cai; a sazonalidade, que se repete em período fixo, aqui o ano; o ciclo, que se repete sem período fixo; e o ruído, o movimento irregular que sobra depois dos outros três. Uma série estacionária mantém média e variância estáveis no tempo, o que não acontece quando há tendência, sazonalidade ou ciclo. A diferença entre valores consecutivos, ou a diferença sazonal, devolve a série a essa condição, e a função de autocorrelação (ACF) e a autocorrelação parcial (PACF) mostram quanto cada valor depende dos anteriores, o que orienta a ordem do modelo.

O preço de uma ação se comporta como série caótica, em que o movimento diário é quase todo ruído e não tem período. No recorte, o retorno diário de PETR4 tem média de 0,056% e desvio padrão de 2,1%, e por isso o grupo mantém curto o horizonte do vetor diário, de 5 e 21 dias úteis.

Dois tratamentos preparam as séries para os modelos. Um buraco no calendário recebe o valor anterior, o seguinte ou a média de uma janela curta, sem que o valor medido seja removido. Além disso, a criação de variáveis (*feature engineering*) transforma as séries brutas em indicadores: o projeto pondera o desvio do PPI pelo volume de combustível vendido e soma esse indicador no tempo para formar um estoque. Os modelos recebem esses indicadores prontos, como variáveis exógenas, e o cálculo está no Diagrama de Solução.

## Modelos

O ARIMA combina uma parte autorregressiva (AR), que usa valores passados, com uma média móvel (MA), que usa erros passados, e aplica a integração (I), isto é, a diferença, até a série ficar estacionária (HYNDMAN; ATHANASOPOULOS, 2021). O SARIMA acrescenta termos sazonais quando a série repete um padrão no ano, e o SARIMAX aceita ainda variáveis exógenas, como o PPI.

O Prophet segue outro caminho e trata a série como soma de uma tendência por partes, de sazonalidades em séries de Fourier, de feriados e de regressores externos (TAYLOR; LETHAM, 2018). Como o ajuste é uma regressão no tempo, o modelo não exige série estacionária, aceita datas faltantes e recebe o `prod` como regressor. Ele funciona bem em séries com tendência e ciclo anual, mas perde força em série caótica, como o preço diário de uma ação. Por isso o Prophet fica no preço semanal e na atividade mensal, e não no fechamento diário de PETR4.

A rede neural recorrente (RNN) devolve a saída do passo anterior à entrada e, assim, guarda memória da sequência. A LSTM (memória de longo e curto prazo) é a forma mais usada dessa rede (HOCHREITER; SCHMIDHUBER, 1997). Neste recorte ela entra só no vetor diário, com dois canais: o fechamento de PETR4 e o `prod` contemporâneo do mesmo pregão.

O TimesFM é um modelo de fundação para séries temporais, desenvolvido pelo Google Research: um transformador do tipo decodificador que lê a série em trechos (*patches*) e prevê os trechos seguintes (DAS et al., 2024). A versão 3.0, de agosto de 2026, tem 330 milhões de parâmetros, foi pré-treinada com mais de um trilhão de pontos de séries temporais e aceita entrada com covariáveis. Neste recorte ele entra só no vetor diário, com PETR4 e o `prod` do treino como covariável passada. A API do TimesFM 3.0 não recebe regressor no horizonte. O código tem licença Apache 2.0, e os pesos da versão 3.0 têm licença não comercial, que permite o uso em estudo acadêmico.

## Trabalhos correlatos

Lopes et al. (2021) estudaram os fatores que moveram a ação PETR4 entre 2009 e 2020, com correlação de Pearson e regressão por mínimos quadrados ordinários. O preço do Brent, o Ibovespa e o volume de produção foram significativos, enquanto a política de preços de combustíveis apareceu só na discussão qualitativa. O trabalho explica o nível da ação, mas não faz previsão nem usa o PPI.

Puentes Márquez et al. (2021) previram a demanda de etanol combustível no Brasil e compararam o ARIMA com redes LSTM de uma e de várias camadas, em versão univariada e multivariada. A LSTM superou o ARIMA mesmo com 180 observações de treino. O trabalho não usa o PPI; no projeto a LSTM do vetor diário recebe o `prod` contemporâneo.

Serrano et al. (2025) previram o consumo de gasolina e de etanol no Brasil com séries de 72, 144 e 263 meses e compararam ARIMA/SARIMA, Holt-Winters, ETS, TBATS, Prophet, Orbit, N-BEATS e TFT. O TBATS teve o menor erro para a gasolina (MAPE de 3,22%) e o Orbit para o etanol (MAPE de 6,77%); como nenhum modelo venceu nos dois combustíveis, o estudo reforça a comparação de mais de uma técnica.

Os três trabalhos usam uma frequência só e não ponderam o PPI pelo volume. O projeto junta dia, semana, mês e trimestre e lê cada alvo contra o `prod` contemporâneo: naive e SARIMAX em todos os vetores; LSTM e TimesFM só no dia; Prophet no preço e na atividade.

# **Diagrama de Solução**

O diagrama de solução apresenta as cinco fases da análise de séries temporais: coleta, pré-processamento, análise exploratória, modelagem e avaliação. O produto desvio × volume diário e o estoque são calculados antes dos modelos, que recebem esses indicadores prontos. Cada alvo entra contra o `prod` (ou `prod_pbr`) do mesmo período. A linha ingênua é a dummy da disciplina; o SARIMAX é o modelo base. A LSTM e o TimesFM 3.0 entram só no vetor diário, com `prod`. O Prophet entra no preço e na atividade com o mesmo regressor. O trimestre fica com naive e SARIMAX (`prod_pbr`).

```
1. COLETA
   base congelada data/gas_flare.sqlite
   âncora: PPI diário (ABICOM), 1.151 datas, corte em 21/09/2026
        |
        v
2. PRÉ-PROCESSAMENTO
   alinhamento ao calendário de dias úteis
   buraco de calendário -> valor vizinho
   conferência da leitura OCR do PPI
   volume diário (ANP: m³ do mês; Petrobras: mil barris por dia)
   produto do dia = desvio em R＄/L × volume diário
   soma dos produtos = estoque
        |
        v
3. ANÁLISE EXPLORATÓRIA E VISUALIZAÇÃO
   tendência, sazonalidade, ciclo e ruído
   estacionariedade e diferença
   ACF e PACF
   correlação cruzada PPI × alvo -> defasagem
   gráfico de linha, de caixa e mapa de calor
        |
        v
4. MODELAGEM (um corte temporal por vetor)
        +-------------+-------------+-------------+
        |             |             |             |
        v             v             v             v
     mercado        preço       atividade     companhia
     dia útil       semana         mês        trimestre
     naive          naive        naive         naive
     SARIMAX        SARIMAX      SARIMAX       SARIMAX
     LSTM           Prophet      Prophet       prod. próprio trim.
     TimesFM 3.0
        |             |             |             |
        +-------------+------+------+-------------+
                             |
                             v
5. AVALIAÇÃO E VALIDAÇÃO
   erro absoluto médio, erro quadrático médio e sua raiz
   gráfico da série real contra a prevista
   horizonte próprio de cada vetor
```

*Diagrama de solução da Etapa 3 — cinco fases e um corte temporal por vetor.*

## Pré-processamento

**PPI.** O cartão da ABICOM traz o desvio em R＄/L e em %, e o percentual é calculado sobre o preço vigente da Petrobras: (preço Petrobras − paridade) / preço Petrobras. A razão entre as duas colunas fica constante entre reajustes, o que confirma essa conta. Em 5 de maio de 2026, por exemplo, a gasolina marca −R＄ 2,22/L e −88%, com o preço Petrobras perto de R＄ 2,52/L e a paridade perto de R＄ 4,74/L; o reajuste até a paridade seria de 88% sobre o preço vigente. Em 2026, o preço Petrobras usado pelo PPI é o preço com desconto das tabelas da Petrobras, enquanto essa coluna existe, ao passo que a série de preço às distribuidoras usa o preço de lista, e por isso as duas séries diferem nesse ano.

**Leitura OCR.** Os valores do PPI saem da imagem da ABICOM por OCR. A revisão da Etapa 2 encontrou leituras erradas, que o pipeline de origem corrigiu contra a imagem, e uma regra passou a comparar cada percentual com o preço de referência dos dez dias anteriores. Quando a divergência passa de 1,5 ponto percentual, a linha vai para conferência manual, marcada na coluna `man_insp`.

**Volume diário.** O desvio em R＄/L entra multiplicado por um volume diário, porque a soma do desvio sem volume não mede o combustível que circulou. Como a base não publica venda diária, os vetores de mercado, preço e atividade tiram o volume diário da venda mensal da ANP, com o volume em m³ vezes 1.000, dividido pelos dias do mês. A ANP soma todos os fornecedores, e a Petrobras responde por cerca de 58% desse volume no 3º trimestre de 2025; por isso o vetor da companhia usa a taxa da própria Petrobras, em mil barris por dia, vezes 159 L por barril. O produto do dia é o desvio vezes esse volume, em R＄, e a série acumulada soma esses produtos. O percentual fica fora da multiplicação, porque a divisão pelo preço já ocorreu. A exógena é sempre do mesmo período do alvo: não há defasagem de PPI × volume.

**Escala e calendário.** A LSTM recebe PETR4 e `prod` normalizados entre 0 e 1 no treino. O SARIMAX, o Prophet e o TimesFM recebem os valores originais. Como o pregão de PETR4 começa em 1º de fevereiro de 2022, as datas de janeiro de 2022 saem do treino do vetor diário.

## Análise exploratória

A leitura de cada vetor separa tendência, sazonalidade, ciclo e ruído, testa a estacionariedade e mede a ACF e a PACF. A EDA preliminar já mostra três padrões. A venda de diesel da ANP tem pico no 3º trimestre em 2022 e em 2023, e a participação da Petrobras no volume da ANP cai cerca de 0,9 ponto percentual por trimestre desde 2022. Em 2026, a variação semanal típica do preço ANP é três a quatro vezes maior que em 2022–2025; como esse preço é média de produtores e importadores e o preço Petrobras fica muito abaixo da paridade nesse ano, uma causa provável é a parcela importada, que muda de semana para semana. Os gráficos, os testes ADF e a correlação cruzada de cada vetor ficam na seção EDA desta etapa.

A correlação cruzada entre o PPI e cada alvo mede a associação linear em cada defasagem, em dias, semanas ou meses; o pico descreve o treino e não identifica quando o alvo responde a um choque. Essa leitura fica na EDA. No modelo, o PPI e o produto desvio × volume entram no **mesmo** período do alvo, sem defasagem.

## Modelagem

Como a base junta dia, semana, mês e trimestre, a proposta separa quatro vetores, para não levar o horizonte de 5 e 21 dias úteis às séries mais lentas. O Quadro 2 apresenta os vetores, os horizontes e as famílias.

Quadro 2 — Vetores, horizontes e famílias

| Vetor | Alvo | Entrada | Horizonte | Modelo |
|---|---|---|---|---|
| Mercado | fechamento de PETR4 | `prod` do mesmo dia | 5 e 21 dias úteis | naive, SARIMAX, LSTM e TimesFM 3.0, todos com `prod` (naive sem `prod`) |
| Preço | preço da ANP e preço da Petrobras | `prod` da mesma semana | 4 e 8 semanas | naive, SARIMAX sazonal e Prophet, com `prod` |
| Atividade | tráfego pesado, indústria, varejo e desemprego | `prod` somado no mês | 1 e 3 meses | naive, SARIMAX sazonal e Prophet, com `prod` |
| Companhia | volume, receita RT&M, lucro bruto e margem | `prod_pbr` do próprio trimestre | 2º trimestre de 2026 | naive e SARIMAX com `prod_pbr`, conferido com o ITR publicado |

O modelo base desta etapa é a linha ingênua e o SARIMAX com ordem de `auto_arima`. O Prophet entra no preço e na atividade com o mesmo `prod` contemporâneo (`add_regressor`). A LSTM e o TimesFM 3.0 entram só no vetor diário, com `prod`. O teste usa o produto observado nesse horizonte, salvo o TimesFM, cuja API só lê `prod` no treino. O termo sazonal entra quando a EDA mostra repetição anual. No vetor da companhia o SARIMAX recebe o `prod_pbr` daquele trimestre e força `d=1`. O 1T25 usa a soma de 1º de janeiro a 31 de março de 2025; o 2T26 usa abril a junho de 2026. A série mensal, com cerca de 55 pontos, fica com o SARIMAX e o Prophet; a trimestral, com 17, fica só com o SARIMAX.

## Avaliação e partição

Em todos os vetores, a avaliação usa o erro absoluto médio, o erro quadrático médio, a raiz desse erro e o gráfico da série real contra a prevista.

A partição usa só a base congelada `data/gas_flare.sqlite`, com corte em 21 de setembro de 2026, sem nenhuma observação nova. O corte é temporal e há um por vetor; no vetor diário, ele é o mesmo para o SARIMAX, a LSTM e o TimesFM 3.0. O teste tem o tamanho do horizonte mais longo, e o horizonte curto é medido dentro desse mesmo teste: 5 dias úteis dentro dos 21, 4 semanas dentro das 8 e 1 mês dentro dos 3.

Alguns pontos ficam de fora para manter as séries alinhadas. O PPI de 21 de setembro de 2026 não entra, porque o pregão de PETR4 termina em 18 de setembro, e o tráfego de agosto de 2026 também não, porque as outras séries mensais param em julho. No resultado da Petrobras, o teste é o 2º trimestre de 2026, último trimestre publicado, cuja receita inclui R＄ 9,7 bilhões de subvenção ao diesel e R＄ 0,8 bilhão à gasolina, fato sem paralelo no treino. Como esse teste tem um ponto só, o erro desse vetor serve apenas como indicação. O Quadro 3 apresenta o recorte de treino e teste de cada vetor.

Os números parciais e os gráficos real × previsto ficam na seção Resultados. A discussão do melhor modelo e da defasagem do PPI fica na Etapa 4.


# **Cronograma**

Até 9 de outubro de 2026 o grupo concluiu a base congelada com âncora no PPI (objetivo específico 1), mediu as defasagens no treino (objetivo 2) e alinhou o recorte: naive e SARIMAX em todos os vetores, LSTM e TimesFM com `prod` só no dia, Prophet com `prod` no preço e na atividade (objetivo 6). A passagem completa no MacBook atualiza o JSON. O Quadro 5 apresenta o cronograma restante, com origem nos objetivos específicos e divisão entre os integrantes.

Quadro 5 — Cronograma restante

| Tarefa | Obj. | Responsável | Início | Término |
|---|---|---|---|---|
| Executar o notebook, gravar as saídas das células e publicar o GitHub | 6 | Ricardo | 06/10/2026 | 12/10/2026 |
| Aplicar o feedback da Etapa 2 quando a nota sair (siglas, citações, template) | 6 | grupo | 13/10/2026 | 19/10/2026 |
| Conferir figuras numeradas, pipeline e entrega da Etapa 3 | 6 | Guilherme L. | 20/10/2026 | 26/10/2026 |
| Comparar as famílias nas tabelas e nos gráficos real × previsto | 6 | Guilherme D. | 27/10/2026 | 02/11/2026 |
| Analisar a defasagem do PPI frente ao erro de previsão (ODS 8 e 11) | 2, 3, 4 | Ricardo | 03/11/2026 | 09/11/2026 |
| Redigir Discussão e Conclusão; Encontro 4 em 10/11, 19h | 6 | Ricardo | 10/11/2026 | 16/11/2026 |
| Preparar slides e roteiro do vídeo (5 a 10 min) | 5, 6 | Guilherme L. | 17/11/2026 | 23/11/2026 |
| Gravar, publicar o vídeo e revisar o notebook final | 6 | grupo | 24/11/2026 | 30/11/2026 |

*Os dias 26/10 e 30/11 são marcos do curso. O Encontro 4 foi marcado para 10/11/2026, às 19h.*


# **EDA e Pré-processamento dos dados**

A base congelada `data/gas_flare.sqlite` alimenta os quatro vetores com o corte de 21 de setembro de 2026. O recorte diário segue o calendário de PETR4, de 1º de fevereiro de 2022 a 18 de setembro de 2026 (1.156 pregões), e o PPI de 21 de setembro fica de fora porque não há fechamento nesse dia. O volume diário dos vetores de mercado, preço e atividade sai da venda mensal da ANP: o m³ do mês vezes 1.000, dividido pelos dias do mês; no vetor da companhia, a taxa da Petrobras em mil barris por dia vira litros com o fator 1.000 × 159 L. O produto do dia é o desvio em R＄/L vezes esse volume, e o estoque soma esses produtos. Como a ANP para em julho de 2026, agosto e setembro repetem a última taxa conhecida, e qualquer buraco de calendário recebe o valor anterior.

Os testes de estacionariedade, a ACF, a PACF e a correlação cruzada usam só a janela de treino de cada vetor. A LSTM recebe PETR4 e `prod` em escala 0–1 ajustada nesse treino. As Figuras 1 a 13d descrevem cada vetor; o recorte de treino e teste está no Quadro 3, e o Quadro 4 resume a exógena contemporânea.

Quadro 3 — Recortes de treino e teste

| Vetor | Treino | Teste | Horizontes |
|---|---|---|---|
| Mercado | 1º fev. 2022 a 19 ago. 2026 | 20 ago. 2026 a 18 set. 2026 (21 dias úteis) | 5 e 21 dias |
| Preço | até a semana encerrada em 19 jul. 2026 | 20 jul. 2026 a 13 set. 2026 (8 semanas) | 4 e 8 semanas |
| Atividade | até abr. 2026 | mai., jun. e jul. 2026 | 1 e 3 meses |
| Companhia | 1º trim. 2022 ao 1º trim. 2026 | 2º trim. 2026 | um trimestre |


In [2]:
# Ambiente e leitura da base congelada
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").is_dir() else cwd.parent
sys.path.insert(0, str(project_root / "src"))

from db_access import read_engine, resolve_database_url
from paths import (
    ensure_output_dirs,
    figures_eda_dir,
    figures_forecasts_dir,
    images_dir,
    results_dir,
)

ENGINE = read_engine(project_root)
ensure_output_dirs(project_root)
IMAGES = images_dir(project_root)
FIGURES_EDA = figures_eda_dir(project_root)
FIGURES_FC = figures_forecasts_dir(project_root)
RESULTS = results_dir(project_root)
print(resolve_database_url(project_root))
device = "cpu"


sqlite:////mnt/opencode/proj_IV/data/gas_flare.sqlite


In [3]:
def sql(q: str) -> pd.DataFrame:
    return pd.read_sql_query(q, ENGINE)

ppi = sql(
    "SELECT report_date, pbr_gas_R, pbr_gas_pct, pbr_die_R, pbr_die_pct "
    "FROM ppi_daily ORDER BY report_date"
)
ppi["report_date"] = pd.to_datetime(ppi["report_date"])
ppi = ppi.set_index("report_date").sort_index()

eq = sql("SELECT tradedate, price_last FROM equity_daily WHERE ticker='PETR4' ORDER BY tradedate")
eq["tradedate"] = pd.to_datetime(eq["tradedate"])
eq = eq.set_index("tradedate").sort_index()["price_last"]

anp_sales = sql("SELECT product, month, volume_m3 FROM anp_fuel_sales ORDER BY month, product")
anp_sales["month"] = pd.to_datetime(anp_sales["month"])
anp_wide = anp_sales.pivot(index="month", columns="product", values="volume_m3")
anp_wide.columns = [f"anp_{c}_m3" for c in anp_wide.columns]

def weekly(table, col):
    d = sql(f"SELECT week_start, avg_price_rs_l AS {col} FROM {table}")
    d["week_start"] = pd.to_datetime(d["week_start"])
    return d.set_index("week_start").sort_index()

price_week = (
    weekly("anp_distributor_weekly_gasoline", "anp_gas")
    .join(weekly("anp_distributor_weekly_diesel", "anp_die"), how="outer")
    .join(weekly("petrobras_distributor_weekly_gasoline", "pbr_gas"), how="outer")
    .join(weekly("petrobras_distributor_weekly_diesel", "pbr_die"), how="outer")
    .sort_index()
)
PRICE_TEST_END = pd.Timestamp("2026-09-13")
week_idx = pd.date_range(price_week.index.min(), PRICE_TEST_END, freq="W-MON")
price_week = price_week.reindex(week_idx).ffill()

abcr = sql("SELECT month, vehicle_type, traffic_index FROM abcr_traffic_index")
abcr["month"] = pd.to_datetime(abcr["month"])
abcr_wide = abcr.pivot(index="month", columns="vehicle_type", values="traffic_index").sort_index()
abcr_wide = abcr_wide.rename(columns={"light": "abcr_light", "heavy": "abcr_heavy", "total": "abcr_total"})

ibge = sql("SELECT indicator_code, month, value FROM ibge_indicators")
ibge["month"] = pd.to_datetime(ibge["month"])
ibge_wide = ibge.pivot(index="month", columns="indicator_code", values="value").sort_index()
ibge_wide = ibge_wide.rename(columns={"6381": "unemp", "8880": "pmc", "8888": "pim"})

fin = sql(
    "SELECT year, quarter_num, diesel_mbpd, gasoline_mbpd, "
    "diesel_revenue_rsmn, gasoline_revenue_rsmn, "
    "rtm_sales_revenue_rsmn, rtm_gross_profit_rsmn, rtm_gross_margin_pct, "
    "extraction_note FROM pbr_quarterly_diesel_gasoline "
    "WHERE year >= 2022 ORDER BY year, quarter_num"
)
fin["quarter_end"] = pd.to_datetime(
    fin.apply(lambda r: pd.Timestamp(int(r.year), int(r.quarter_num) * 3, 1) + pd.offsets.MonthEnd(0), axis=1)
)
fin = fin.set_index("quarter_end").sort_index()

print("ppi", len(ppi), ppi.index.min().date(), ppi.index.max().date())
print("petr4", eq.notna().sum(), eq.index.min().date(), eq.index.max().date())
print("anp meses", anp_wide.dropna(how="all").shape[0], anp_wide.index.min().date(), anp_wide.index.max().date())
print("preço semanas", len(price_week))
print("fin trimestres 2022+", len(fin))


ppi 1151 2022-01-05 2026-09-21
petr4 1156 2022-02-01 2026-09-18
anp meses 55 2022-01-01 2026-07-01
preço semanas 245
fin trimestres 2022+ 18


In [4]:
LITERS_PER_M3 = 1000.0
LITERS_PER_MBPD = 1000.0 * 159.0  # mil barris/dia → L/dia

MKT_TRAIN_END = pd.Timestamp("2026-08-19")
MKT_TEST_END = pd.Timestamp("2026-09-18")
PRICE_TRAIN_END = pd.Timestamp("2026-07-19")
ACT_TRAIN_END = pd.Timestamp("2026-04-01")
ACT_TEST_END = pd.Timestamp("2026-07-01")

daily_idx = eq.index
vol = pd.DataFrame(index=daily_idx)
vol["month"] = vol.index.to_period("M").to_timestamp()
vol = vol.join(anp_wide, on="month")
vol["anp_gasolina_c_m3"] = vol["anp_gasolina_c_m3"].fillna(anp_wide["anp_gasolina_c_m3"].dropna().iloc[-1])
vol["anp_diesel_m3"] = vol["anp_diesel_m3"].fillna(anp_wide["anp_diesel_m3"].dropna().iloc[-1])
days = vol.index.to_series().dt.days_in_month
vol["vol_gas_l"] = vol["anp_gasolina_c_m3"] * LITERS_PER_M3 / days
vol["vol_die_l"] = vol["anp_diesel_m3"] * LITERS_PER_M3 / days

pbr_daily = pd.DataFrame(index=daily_idx)
pbr_daily["qend"] = pd.DatetimeIndex(pbr_daily.index.to_period("Q").end_time).normalize()
pbr_daily = pbr_daily.join(fin[["diesel_mbpd", "gasoline_mbpd"]], on="qend")
pbr_daily[["diesel_mbpd", "gasoline_mbpd"]] = pbr_daily[["diesel_mbpd", "gasoline_mbpd"]].ffill()

daily = pd.DataFrame({"petr4": eq}, index=daily_idx)
daily = daily.join(ppi[["pbr_gas_R", "pbr_die_R", "pbr_gas_pct", "pbr_die_pct"]])
daily[["pbr_gas_R", "pbr_die_R", "pbr_gas_pct", "pbr_die_pct"]] = daily[
    ["pbr_gas_R", "pbr_die_R", "pbr_gas_pct", "pbr_die_pct"]
].ffill()
daily["vol_gas_l"] = vol["vol_gas_l"]
daily["vol_die_l"] = vol["vol_die_l"]
daily["prod"] = daily["pbr_gas_R"] * daily["vol_gas_l"] + daily["pbr_die_R"] * daily["vol_die_l"]
daily["stock"] = daily["prod"].cumsum()
daily["prod_pbr"] = (
    daily["pbr_gas_R"] * pbr_daily["gasoline_mbpd"] * LITERS_PER_MBPD
    + daily["pbr_die_R"] * pbr_daily["diesel_mbpd"] * LITERS_PER_MBPD
)
daily["stock_pbr"] = daily["prod_pbr"].cumsum()

mkt_train = daily.loc[:MKT_TRAIN_END]
price_train = price_week.loc[:PRICE_TRAIN_END]
print("treino diário", mkt_train.index.min().date(), "→", mkt_train.index.max().date(), "n=", len(mkt_train))
print("produto diário (média R$ mi)", round(mkt_train["prod"].mean() / 1e6, 1))


treino diário 2022-02-01 → 2026-08-19 n= 1135
produto diário (média R$ mi) -104.2


In [5]:
import sys
import warnings
warnings.filterwarnings("ignore")

if sys.platform == "win32":
    print("Neste Windows o ADF do statsmodels não completa (numpy.linalg).")
    print("Os números do treino estão no texto. No MacBook M4 esta célula reproduz o ADF.")
else:
    from statsmodels.tsa.stattools import adfuller, ccf

    def adf_row(name, s):
        x = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if len(x) < 20:
            return {"série": name, "n": int(len(x)), "ADF": np.nan, "p": np.nan, "lags": np.nan}
        stat, p, lags, *_ = adfuller(x.to_numpy(dtype=float), autolag="AIC")
        return {"série": name, "n": int(len(x)), "ADF": round(float(stat), 3), "p": round(float(p), 4), "lags": int(lags)}

    def best_ccf(x, y, max_lag):
        aligned = pd.concat([x, y], axis=1).dropna()
        vals = ccf(aligned.iloc[:, 0], aligned.iloc[:, 1], adjusted=False)[: max_lag + 1]
        k = int(np.argmax(np.abs(vals)))
        return k, float(vals[k])

    adf_daily = pd.DataFrame([
        adf_row("PETR4", mkt_train["petr4"]),
        adf_row("PETR4 d1", mkt_train["petr4"].diff()),
        adf_row("PPI gasolina R$/L", mkt_train["pbr_gas_R"]),
        adf_row("PPI diesel R$/L", mkt_train["pbr_die_R"]),
        adf_row("produto do dia", mkt_train["prod"]),
        adf_row("produto do dia d1", mkt_train["prod"].diff()),
    ])
    display(adf_daily)

    print("CCF PETR4 ← PPI (treino, 0–21 pregões)")
    for col, lab in [("pbr_gas_R", "gasolina R$"), ("pbr_die_R", "diesel R$"), ("prod", "produto")]:
        k, v = best_ccf(mkt_train[col], mkt_train["petr4"], 21)
        print(f"  {lab}: defasagem {k}, ccf {v:.3f}")


,série,n,ADF,p,lags
0,PETR4,1135,-1.857,0.3526,0
1,PETR4 d1,1134,-24.732,0.0000,1
2,PPI gasolina R$/L,1135,-2.852,0.0512,16
3,PPI diesel R$/L,1135,-2.140,0.2288,14
4,produto do dia,1135,-1.624,0.4707,19
5,produto do dia d1,1134,-8.849,0.0000,18


CCF PETR4 ← PPI (treino, 0–21 pregões)
  gasolina R$: defasagem 1, ccf -0.553
  diesel R$: defasagem 0, ccf -0.573
  produto: defasagem 0, ccf -0.592


## Vetor de mercado

O fechamento de PETR4 no treino (1.135 pregões até 19 de agosto de 2026) não é estacionário (ADF −1,86), mas a primeira diferença é (ADF −32,9). O retorno diário nesse treino tem média 0,046% e desvio-padrão 2,1%, o que mantém curtos os horizontes de 5 e 21 dias úteis.

A Figura 1 percorre os 1.156 pregões até 18 de setembro de 2026. A série sobe no recorte e oscila sem um ciclo anual visível, o que combina com o ADF no nível e com a ausência de sazonalidade semanal no SARIMAX diário.

![Fechamento de PETR4](../figures/eda/eda_mercado_petr4.svg)

*Figura 1 — Fechamento de PETR4, 1º de fevereiro de 2022 a 18 de setembro de 2026.*

A Figura 2 mostra a ACF e a PACF do treino. A ACF do nível decai devagar, padrão de raiz unitária, e a PACF da primeira diferença corta cedo. Esse desenho leva o `auto_arima` a (0,1,0): o próximo fechamento é o último mais um ruído, sem termo AR ou MA longo.

![ACF e PACF de PETR4 no treino](../figures/eda/eda_mercado_acf.svg)

*Figura 2 — ACF e PACF de PETR4 no treino (até 19 de agosto de 2026).*

O desvio do PPI em R＄/L já é estacionário no treino (gasolina ADF −3,49; diesel −3,53). O produto diário desvio × volume ANP ainda traz tendência (ADF −2,11) e estaciona na diferença (ADF −9,43). A correlação cruzada com PETR4 é contemporânea e negativa: −0,55 com o PPI da gasolina, −0,57 com o do diesel e −0,59 com o produto do dia, todas no lag 0. O modelo usa esse produto no mesmo dia do fechamento, inclusive no teste.

A Figura 3 registra o desvio em R＄/L, já estacionário, com o aprofundamento de 2026, quando o cartão da ABICOM usa o preço Petrobras com desconto. A Figura 4 isola o produto diário, que ainda carrega a tendência do volume, e a Figura 5 acumula esses produtos no estoque. O produto entra no modelo como exógena já calculada, e o estoque permanece descritivo.

![Desvio PPI em R＄/L](../figures/eda/eda_ppi_desvio.svg)

*Figura 3 — Desvio do PPI em R＄/L, gasolina e diesel, no recorte diário.*

![Produto diário desvio × volume](../figures/eda/eda_indicador_prod.svg)

*Figura 4 — Produto diário desvio × volume ANP.*

![Estoque acumulado](../figures/eda/eda_indicador_stock.svg)

*Figura 5 — Estoque acumulado do produto diário.*


## Vetor de preço

As séries semanais de preço ANP no treino (237 semanas) não são estacionárias no nível (gasolina ADF −2,58; diesel −2,01) e estacionam na primeira diferença. O preço de lista da Petrobras, reindexado às segundas e preenchido para a frente nas semanas sem PDF novo, também fica no nível (gasolina −2,65; diesel −2,25). A ACF do diesel ANP no treino decai devagar, o que pede termo sazonal anual (`s=52`) no SARIMAX e no Prophet.

A Figura 6 junta as quatro séries semanais. O preço ANP se move semana a semana, e o preço de lista da Petrobras avança em degraus, parado nas semanas sem PDF novo. A linha ingênua cola nesse degrau. O SARIMAX e o Prophet recebem o `prod` da semana e, no teste, reagem ao produto contemporâneo. A Figura 7 traz a ACF e a PACF do diesel ANP: a ACF lenta no nível confirma a diferença, e o período de 52 semanas justifica o termo sazonal, ainda que a busca `auto_arima` tenha devolvido (0,1,0)(0,0,0)[52] para o diesel.

A correlação cruzada com o PPI da semana aponta máximo em defasagem 1 na gasolina ANP (ccf −0,57) e defasagem 2 no diesel ANP (ccf −0,30); no preço Petrobras a correlação é mais fraca e o máximo cai no horizonte 8. O desvio-padrão semanal do diesel ANP em 2026 é 0,55 R＄/L, cerca de 3,4 vezes o de 2025 (0,16 R＄/L), enquanto 2024 ficou quase estável (0,043 R＄/L). Esses picos de ccf descrevem o treino. O modelo usa o PPI da **mesma** semana do preço, no treino e no teste.

![Preço semanal ANP e Petrobras](../figures/eda/eda_preco.svg)

*Figura 6 — Preço semanal ANP e preço de lista Petrobras, gasolina e diesel.*

![ACF e PACF do diesel ANP](../figures/eda/eda_preco_acf.svg)

*Figura 7 — ACF e PACF do diesel ANP no treino (237 semanas).*


## Vetor de atividade

O painel mensal de treino tem 51 a 52 pontos até abril de 2026. Nenhuma das séries de tráfego pesado, indústria, varejo, desemprego ou venda de diesel da ANP rejeita a raiz unitária no nível, e a primeira diferença de tráfego e de indústria também fica acima do limiar usual, com amostra curta; por isso o SARIMAX mensal entra com diferença e sazonalidade anual (`s=12`) quando a busca de ordem confirma o termo.

As Figuras 8 a 11 mostram as quatro séries mensais no recorte de treino e de teste. Tráfego pesado, indústria e varejo repetem um ciclo de cerca de doze meses, o que pede o termo sazonal e justifica o Prophet com sazonalidade anual. O desemprego anda mais devagar, e o SARIMAX (0,1,2)(1,0,0)[12] é o desenho que a busca devolve. A Figura 12 decompõe a venda mensal de diesel da ANP: o pico do terceiro trimestre aparece em 2022 e em 2023, e a média móvel de 12 meses sobe no recorte. Essa sazonalidade de volume é o motivo de o modelo mensal não ficar só na primeira diferença.

A correlação cruzada com a média mensal do PPI do diesel é fraca: −0,19 no tráfego pesado e −0,13 na indústria, ambas no lag 0, e o varejo tem máximo −0,09 no lag 5, enquanto o desemprego correlaciona +0,15 no contemporâneo. O modelo mensal usa o produto desvio × volume somado **no mesmo mês** do alvo, no treino e no teste.

A participação da Petrobras no volume de diesel da ANP sai de 68,2% no 1º trimestre de 2022 para 63,4% no 3º trimestre de 2025. O tráfego de agosto de 2026 fica fora do alinhamento porque é o único mês ABCR sem par no IBGE.

![Tráfego pesado ABCR](../figures/eda/eda_atividade_heavy.svg)

*Figura 8 — Tráfego pesado ABCR, índice mensal.*

![Indústria PIM-PF](../figures/eda/eda_atividade_pim.svg)

*Figura 9 — Produção industrial (PIM-PF), índice mensal.*

![Varejo PMC](../figures/eda/eda_atividade_pmc.svg)

*Figura 10 — Volume de vendas no varejo (PMC), índice mensal.*

![Desemprego PNAD](../figures/eda/eda_atividade_unemp.svg)

*Figura 11 — Taxa de desemprego (PNAD Contínua), pontos percentuais.*

![Venda mensal de diesel ANP e tendência de 12 meses](../figures/eda/eda_decomp_diesel_anp.svg)

*Figura 12 — Venda mensal de diesel ANP (m³) e tendência de 12 meses.*


## Vetor da companhia

Há 17 trimestres completos no treino (1T22 a 1T26) e um ponto de teste (2T26). Com menos de 20 observações o ADF não se aplica, de modo que a série trimestral permanece descritiva e o SARIMAX entra só como leitura de um passo à frente. O 2T26 publicado tem diesel 742 mbpd, gasolina 404 mbpd, receita de diesel R＄ 38.262 milhões e de gasolina R＄ 15.209 milhões, vendas RT&M R＄ 163.193 milhões, lucro bruto R＄ 26.130 milhões e margem 16,01%. A nota 4 do ITR inclui R＄ 9.737 milhões de subvenção ao diesel e R＄ 816 milhões à gasolina, fato sem paralelo no treino, e por isso o erro desse vetor vale como indicação.

A Figura 13 mostra o volume Petrobras de diesel e de gasolina; as Figuras 13b, 13c e 13d separam receita, lucro bruto e margem de RT&M. O volume de diesel e de gasolina anda em degrau; a receita e o lucro oscilam entre trimestres. A margem cai de 20,27% no 1T26 para 16,01% no 2T26, no mesmo trimestre da subvenção. O SARIMAX com `prod_pbr` contemporâneo aponta 23,99% nesse ponto, na direção oposta ao ITR: o produto do próprio trimestre não descreve a queda de margem ligada à nota 4.

![Volume Petrobras](../figures/eda/eda_companhia_volume.svg)

*Figura 13 — Volume Petrobras de diesel e gasolina, mil barris por dia, 1º trimestre de 2022 ao 2º trimestre de 2026.*

![Receita RT&M](../figures/eda/eda_companhia_receita.svg)

*Figura 13b — Receita RT&M, R＄ milhões, 1º trimestre de 2022 ao 2º trimestre de 2026.*

![Lucro bruto RT&M](../figures/eda/eda_companhia_lucro.svg)

*Figura 13c — Lucro bruto RT&M, R＄ milhões, 1º trimestre de 2022 ao 2º trimestre de 2026.*

![Margem bruta RT&M](../figures/eda/eda_companhia_margem.svg)

*Figura 13d — Margem bruta RT&M da Petrobras, 1º trimestre de 2022 ao 2º trimestre de 2026.*

O Quadro 4 resume a exógena de cada vetor: o pico de ccf descreve o treino, e o modelo usa o produto no mesmo período do alvo.

Quadro 4 — Exógena no mesmo período do alvo (o ccf do treino só descreve; o modelo não defasa)

| Vetor | Alvo | Exógena | Pico de ccf no treino | Uso no modelo |
|---|---|---|---|---|
| Mercado | PETR4 | `prod` do dia | 0 pregões | mesmo dia, inclusive no teste |
| Preço | ANP gasolina | `prod` da semana | 1 semana | mesma semana, inclusive no teste |
| Preço | ANP diesel | `prod` da semana | 2 semanas | mesma semana, inclusive no teste |
| Atividade | tráfego, PIM, PMC, desemprego | produto somado no mês | 0 a 5 meses, ccf em módulo menor que 0,20 | mesmo mês, inclusive no teste |
| Companhia | volume e RT&M | produto PPI × volume do próprio trimestre | — | mesmo trimestre (ex.: 1º jan.–31 mar. = 1T) |


### Revisão da base para a Etapa 4

A leitura preserva o SQLite. O hash SHA-256 efetivo, a impressão digital do arquivo, é verificado antes dos experimentos. O RT&M tem 82 trimestres no calendário, dos quais 26 trazem as contas do segmento. As exposições semanais usam segunda-feira a domingo; a soma mensal e a trimestral mantêm suas unidades. A auditoria de consistência interna não substitui a revisão das fontes originais.


### Base complementar: PETR4 e BOVA11

O BOVA11, fundo de índice que replica o Ibovespa, é o referencial de mercado deste ensaio. A subtração usa peso unitário e não estima beta nem alfa. Os dois papéis entram pelos fechamentos COTAHIST sem ajuste por proventos, de modo que os alvos medem retorno de preço, não retorno total com distribuições.

Os arquivos originais compartilham 1.156 pregões nas mesmas datas; a coincidência das duas séries incompletas não estabelece cobertura completa. O overlay separado restaura o pregão de 17 de novembro de 2023 a partir do COTAHIST importado no Dado (`tbl_cotahist`, arquivo `COTAHIST_D17112023`): PETR4 a R＄ 36,71 e BOVA11 a R＄ 121,10. O painel reparado tem 1.157 sessões, de 1º de fevereiro de 2022 a 18 de setembro de 2026, conferidas contra o calendário independente da B3. O SQLite do curso e o suplemento original permanecem inalterados; o overlay vale só para este ensaio adicional.

Os preços de ações não recebem preenchimento para frente. Retornos diários e defasagens do PPI usam o pregão anterior da bolsa, antes do filtro da amostra. A exposição trimestral usa o volume mensal da ANP no próprio mês e não substitui um mês futuro nem o último disponível quando o volume falta. As Tabelas 5 e 6 e as Figuras 37 a 39 reportam este ensaio. A conferência de hashes e de identidades fica na reconstrução; a auditoria de fonte independente permanece distinta da consistência interna.


In [6]:
from etapa4_data import CURRENT_FREEZE, load_frames_at, sqlite_file
from etapa4_features import add_exposure_variants
frames4 = add_exposure_variants(load_frames_at(sqlite_file(CURRENT_FREEZE, project_root), project_root))
display(frames4['price_week'][['prod_gas_mn', 'prod_die_mn']].tail())
print('Semanas: segunda-feira a domingo; exposição contemporânea em milhões de reais.')


,prod_gas_mn,prod_die_mn
2026-08-10,-129.628116,-472.071726
2026-08-17,-159.840247,-558.536252
2026-08-24,-167.070500,-527.392631
2026-08-31,-163.016394,-609.417825
2026-09-07,-135.416622,-672.217298


Semanas: segunda-feira a domingo; exposição contemporânea em milhões de reais.


# **Modelos**

### Referência histórica — Etapa 3

*(Os parágrafos desta seção descrevem as famílias e os recortes da Etapa 3, gravados em JSON. Não usam o protocolo de um passo nem a dummy contemporânea desta etapa.)*

O modelo base desta etapa é a linha ingênua, que repete o último valor de treino, e o SARIMAX com ordem de `auto_arima` no mesmo recorte. A ordem do SARIMAX sai de `auto_arima` no treino. O vetor diário entra sem sazonalidade anual, salvo a ACF pedir a semana (`s=5`); preço usa `s=52` e atividade `s=12`. A exógena é o `prod` (ou `prod_pbr`) do mesmo período do alvo. O teste usa o produto observado nesse horizonte, salvo o TimesFM.

A LSTM é uma rede PyTorch só no vetor diário, com PETR4 e `prod` contemporâneo em escala 0–1 ajustada no treino. A janela de PETR4 é o histórico; o último passo de `prod` é o produto do mesmo pregão que se prevê. A grade cobre o tamanho da janela (21 ou 63 pregões), com taxa de aprendizado 10⁻³, 40 épocas, 32 unidades e lote 64, e a validação é o último bloco de 21 pregões do treino. O TimesFM 3.0 (`google/timesfm-3.0-pytorch`) roda nesse vetor, sem treino nesta base, com PETR4 e `prod` do treino como covariável passada; os pesos baixam na execução para o cache do Hugging Face. A série mensal e a trimestral são curtas para a LSTM e para o TimesFM.

O Prophet entra nos vetores de preço e atividade, com sazonalidade anual e o mesmo `prod` contemporâneo do SARIMAX. O vetor da companhia usa o SARIMAX de um passo com `prod_pbr` (PPI × volume Petrobras daquele trimestre) e diferenciação `d=1`, conferido com o 2T26 publicado. A linha ingênua entra na tabela como dummy da disciplina.

Os ajustes gravam métricas em `results/etapa3_metrics.json` e as previsões em `results/etapa3_forecasts.json`. Neste Windows o OpenBLAS interrompe o SARIMAX, o Prophet e o TimesFM, de modo que a LSTM e a linha ingênua rodam neste kernel e as demais famílias executam no MacBook M4 (`bash run_etapa3_mac.sh`). A célula seguinte lê o JSON já gravado, para o notebook mostrar a tabela sem reajustar.

No Mac (darwin, arm64) o `auto_arima` devolve PETR4 em (0,1,0), ANP gasolina em (1,0,0)(0,0,0)[52], lista Petrobras gasolina em (0,1,0)(1,0,0)[52] e as demais séries semanais em (0,1,0)(0,0,0)[52]; no painel mensal o termo sazonal entra com `D=1` ou `P=1` e `s=12`. O TimesFM 3.0 roda no backend MLX. No 2T26 o SARIMAX entra com `prod_pbr` contemporâneo e `d=1` (grade statsmodels neste Windows): diesel (2,1,2), gasolina (0,1,2), receita (0,1,2), lucro bruto (0,1,2) e margem (0,1,2). As Tabelas 1 a 4 e as Figuras 14 a 27 reúnem MAE, MSE, RMSE e o gráfico real × previsto na seção Resultados; a discussão do melhor modelo fica para a Etapa 4.


In [7]:
# Tabela gravada em results/etapa3_metrics.json.
# Execute este bloco no kernel proj-iv. Ele só lê o JSON e mostra a tabela.
# Neste Windows o OpenBLAS aborta numpy.linalg se o ajuste rodar de novo.
#
# Para reajustar, fora do notebook, no diretório do projeto:
#   bash run_etapa3_mac.sh                      # MacBook M4: recorte congelado
#   python src/etapa3_models.py --only naive,lstm  # este Windows
import json

from IPython.display import display
from paths import forecasts_path, metrics_path

mp = metrics_path(project_root)
fp = forecasts_path(project_root)
payload = json.loads(mp.read_text(encoding="utf-8"))
tab = pd.DataFrame(payload["rows"])
tab = tab[tab["model"].isin(["naive", "sarimax", "prophet", "lstm", "timesfm_cov"])].copy()
prefer = ["vector", "target", "model", "horizon", "MAE", "MSE", "RMSE", "order", "window", "note"]
cols = [c for c in prefer if c in tab.columns] + [c for c in tab.columns if c not in prefer]
display(tab[cols])
print("métricas em", mp)
print("previsões em", fp, "existe" if fp.exists() else "ausente")


,vector,target,model,horizon,MAE,MSE,RMSE,order,window,note,seasonal_order,aic
0,mercado,PETR4,naive,5d,1.364000,1.946680,1.395235,NaN,NaN,NaN,NaN,NaN
1,mercado,PETR4,naive,21d,3.744286,18.606300,4.313502,NaN,NaN,NaN,NaN,NaN
2,preço,ANP gasolina,naive,4w,0.094387,0.010301,0.101495,NaN,NaN,NaN,NaN,NaN
3,preço,ANP gasolina,naive,8w,0.121302,0.022146,0.148815,NaN,NaN,NaN,NaN,NaN
4,preço,ANP diesel,naive,4w,0.398800,0.299569,0.547329,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
61,companhia,margem RT&M,sarimax,2Q26,7.976966,63.631984,7.976966,"[0, 1, 2]",NaN,prod_pbr contemporâneo (PPI × volume do própri...,None,70.613918
62,mercado,PETR4,lstm,5d,1.254363,2.185182,1.478236,NaN,21.0,NaN,NaN,NaN
63,mercado,PETR4,lstm,21d,4.943810,33.279070,5.768801,NaN,21.0,NaN,NaN,NaN
64,mercado,PETR4,timesfm_cov,5d,1.355001,1.918122,1.384963,NaN,NaN,NaN,NaN,NaN


métricas em /mnt/opencode/proj_IV/results/etapa3_metrics.json
previsões em /mnt/opencode/proj_IV/results/etapa3_forecasts.json existe


### Etapa 4 — avaliação corrigida

A hipótese continua a da etapa anterior: o desvio de paridade de combustíveis ponderado pelo volume acompanha o alvo quando se controla a história recente da própria série. Esse produto entra como `prod` no mercado e como `prod_pbr` na companhia; o nome não é produção. Para o teste, o grupo usou a regressão Ridge, uma regressão linear que penaliza coeficientes grandes e, com isso, reduz o risco de colar demais no treino. A Ridge entra com intercepto, com os valores já observados do alvo um e dois períodos antes (defasagens 1 e 2) e com padronização calculada só no treino. O parâmetro de penalização α escolhe-se em {0,1; 1; 10; 100} em blocos internos anteriores à avaliação; no ensaio trimestral descritivo, α = 1. Dentro de cada bloco de datas, os coeficientes ficam fixos, e as defasagens observadas se atualizam a cada nova data.

A comparação ingênua desta etapa é a dummy contemporânea, a previsão que copia o último valor já conhecido: ŷ[t] = y[t−1]. Ela usa a mesma informação observada que a Ridge, e por isso serve de referência justa. A dummy fixa da Etapa 3, que não se atualizava dentro do horizonte, permanece só como referência histórica. Os rótulos 21d, 8w e 3m nomeiam blocos de avaliação, conjuntos de datas em que se mede o erro de um passo, e não previsões de 21 pregões, oito semanas ou três meses de uma só vez. Por isso os erros de um passo da Etapa 4 não se comparam diretamente aos erros de vários passos da Etapa 3.

Cada candidato e o respectivo controle usam as mesmas datas de treino e de avaliação, para que um dado faltante não encolha em silêncio a amostra de um dos lados. A exposição simples, o produto do desvio de paridade pelo volume, compara-se com `y_only`, a Ridge que usa só a história do alvo. A exposição mais sazonalidade compara-se com `y_seasonal`, a mesma Ridge com o ciclo do calendário e sem a exposição. Quando há um contexto extra, o controle é esse contexto sem a exposição. O critério exploratório dos ensaios diários, semanais e mensais pede três coisas ao mesmo tempo: a raiz do erro quadrático médio (RMSE) mediana cai pelo menos 5%; o RMSE é menor na maioria dos mesmos blocos; o erro absoluto médio (MAE) mediano não aumenta. A dummy atualizada aparece numa coluna à parte, porque um RMSE menor que o da dummy não prova que a exposição adiciona informação além da história do alvo. Quando duas especificações ficam até 2% acima do menor RMSE, o estudo fica com a mais simples. Menor erro observado, evidência exploratória e critério atendido permanecem campos distintos.

As semanas de preço começam na segunda-feira e terminam no domingo. A exposição usa o recorte `closed="left", label="left"` do pandas, equivalente a esses intervalos. A sazonalidade mensal entra como a defasagem 12, o mesmo mês do ano anterior. As exposições são contemporâneas: o valor do regressor na data t entra na previsão de t. As defasagens 0 a 8 incluem o lag zero e, por isso, não demonstram que o desvio de paridade antecipe o alvo, nem um atraso causal de oito semanas. A avaliação temporal segue a distinção entre origem, a data em que a previsão é feita, e horizonte em Hyndman e Athanasopoulos (2021).

No ensaio financeiro, o grupo comparou persistência (copiar o último trimestre), persistência sazonal com período 4 (copiar o mesmo trimestre do ano anterior), uma grade limitada de ARIMA e SARIMA trimestrais e três suavizações exponenciais aditivas: nível local, tendência amortecida e tendência amortecida com sazonalidade trimestral. ARIMA é a média móvel integrada auto-regressiva; SARIMA acrescenta o bloco sazonal. São 38 métodos e duas janelas de história (a série inteira até a origem, ou só os 40 trimestres anteriores), o que dá 76 configurações por alvo. A seleção usa os doze trimestres do 1º trimestre de 2019 ao 4º trimestre de 2021. Travar, neste texto, significa escolher o método e a janela com esses trimestres e não os trocar quando a avaliação passa a 2022–2026; a avaliação posterior mede uma escolha já feita, sem o benefício de ter visto o futuro. O 2º trimestre de 2026 entra separado, como ponto legado já inspecionado. Persistência e persistência sazonal não medem o benefício da história longa, porque as previsões coincidem nas duas janelas. ARIMA, SARIMA e as suavizações ajustam cada janela de treino. A padronização usa só o passado da origem, e a previsão volta à unidade original.

O diagnóstico financeiro contra a persistência usa o RMSE agrupado, a raiz do erro quadrático médio calculada sobre todos os trimestres da janela juntos, o MAE médio e a contagem de origens em que o erro absoluto é menor. Esse diagnóstico não é a regra da mediana por bloco dos ensaios diários, semanais e mensais, e não reabre a seleção depois de 2021. Com a escolha de 2019–2021, a receita fica em persistência com história completa; o lucro, em SARIMA(0,1,0)×(1,0,0)₄; a margem, em SARIMA(0,0,0)×(0,0,0)₄, que estima um nível médio da série. O resíduo da extensão financeira é o valor realizado menos essa previsão-base já escolhida na origem. A correção estima, por mínimos quadrados ordinários, um intercepto e um coeficiente para uma exposição, e exige pelo menos oito pares anteriores; com menos pares o estado é histórico insuficiente, e a linha não conta como correção ajustada.

Os ensaios de venda usam o volume contemporâneo na própria exposição e, por isso, permanecem descritivos: o regressor contém parte do que se pretende explicar. A escolha entre várias especificações nos mesmos blocos é seleção exploratória, não confirmação em uma amostra que o grupo não usou para escolher. A viabilidade ex ante, prever só com informação que já teria sido publicada na origem, não foi executada.


### Etapa 4 — retorno de PETR4 em relação ao BOVA11

O ensaio define dois alvos a partir dos fechamentos *P* (PETR4) e *B* (BOVA11). `prev(t)` é o pregão anterior da bolsa e `b(q)` é o último pregão antes do trimestre *q*:

```text
rP[t] = P[t] / P[prev(t)] - 1
rB[t] = B[t] / B[prev(t)] - 1
delta_r[t] = rP[t] - rB[t]
D_qtd[t] = (P[t] / P[b(q)] - 1) - (B[t] / B[b(q)] - 1)
E_f[t] = PPI_f[t] * volume_m3_f[month(t)] * 1000 / days_in_month(t) / 1e6
C_f_qtd[t] = sum(E_f[s] for exchange sessions s in the quarter through t)
```

`D_qtd` é a diferença entre retornos compostos em separado, não a soma dos diferenciais diários. O primeiro pregão do trimestre já inclui o retorno desde a base do trimestre anterior. As somas de exposição são com sinal e só nos pregões; não são receita da companhia nem prejuízo financeiro realizado.

Na dimensão diária, o controle `a_history` usa as defasagens 1 e 2 de `delta_r`. Cada um dos cinco candidatos acrescenta o PPI de gasolina e o de diesel em uma única defasagem, de 1 a 5 pregões, nas colunas `pbr_gas_R` e `pbr_die_R`, sem multiplicar por volume. Zero e persistência atualizada entram como previsões de referência. A avaliação usa seis blocos de 21 pregões, de 19 de fevereiro a 19 de agosto de 2026 (126 observações pontuadas).

Na dimensão trimestral, o estudo compara `b_clock`, `b_clock_prod`, `b_history_clock`, `b_history_clock_prod` e `b_history_clock_fuels`. O relógio é o índice de pregão dentro do trimestre e o seu quadrado. `b_history_clock` acrescenta as defasagens 1 e 2 do próprio `D_qtd` intra-trimestre, com estado inicial zero antes do trimestre. As extensões de exposição total e de combustíveis separados usam esse mesmo controle de história; `b_clock_prod` usa `b_clock` como controle. Zero e persistência com reset entram à parte: a persistência prevê zero no primeiro pregão e, depois, o `D_qtd` observado anterior. A avaliação cobre 2025Q1–2026Q2, seis trimestres completos e 372 observações. O 1º trimestre de 2022 fica de fora porque a base não está disponível. O 3º trimestre de 2026, parcial no corte, não entra como trimestre completo.

Nas duas dimensões a regressão é Ridge, com padronização só no treino e α escolhido em 0,1; 1; 10 e 100 apenas com dados de treino. Os coeficientes permanecem fixos dentro de cada bloco de avaliação; as defasagens observadas do alvo se atualizam dentro do bloco.

O critério incremental pede, ao mesmo tempo, RMSE mediano pelo menos 5% menor, vitória em pelo menos quatro dos seis blocos e MAE mediano que não aumente, sempre contra o controle emparelhado. A regra de 2% escolhe menos regressores entre as especificações a até 2% do menor RMSE; complexidade igual deixa empate sem resolução. Menor erro observado, escolha por simplicidade e atendimento ao critério incremental permanecem campos distintos.

A exposição trimestral é contemporânea e descritiva. O estudo de PPI defasado também não audita o horário de divulgação. Nenhuma das duas dimensões estabelece viabilidade ex ante, causalidade nem confirmação independente depois da seleção.


In [8]:
# Reprodução integral a partir do SQLite; não depende apenas dos JSON existentes.
import io
import json
from contextlib import redirect_stdout
from etapa4_run import main as rebuild_etapa4
from etapa4_delivery import (load, target_table, review_table, interpretation,
                             finance_locked_table, finance_history_table,
                             finance_sensitivity_table, finance_sensitivity_official_table,
                             finance_eval_ranking_table, block_chart, incremental_table)
from IPython.display import Markdown, SVG, display

with redirect_stdout(io.StringIO()) as run_log:
    rebuild_etapa4()
rebuild = load("etapa4_rebuild")
assert rebuild["sha256_actual"] == rebuild["sha256_expected"]
assert rebuild["sqlite_unchanged"] and rebuild["etapa3_unchanged"]
print("Experimentos reconstruídos do SQLite:", len(rebuild["modules_rebuilt"]))
print("SHA-256 efetivo:", rebuild["sha256_actual"])
print("falhas financeiras:", load("etapa4_finance_baseline").get("n_failures"))


Experimentos reconstruídos do SQLite: 11
SHA-256 efetivo: b314283c8e711becd802d5162be38b1c47052e9603d8bbf81345916fc2bf35ba
falhas financeiras: 0


In [9]:
import hashlib
import importlib.metadata
import json
import platform
import sys
from pathlib import Path

import etapa4_petr4_bova11 as bova

def _file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

data_dir = project_root / "data"
sqlite_path = data_dir / "gas_flare.sqlite"
supplement_path = data_dir / "etapa4_bova11_20261018T152955Z.sqlite"
overlay_path = data_dir / "etapa4_equity_overlay_2023-11-17.json"
sha_before = {
    "course_sqlite": _file_sha256(sqlite_path),
    "bova11_supplement": _file_sha256(supplement_path),
    "overlay": _file_sha256(overlay_path),
}

bova_report = bova.rebuild(project_root)

sha_after = {
    "course_sqlite": _file_sha256(sqlite_path),
    "bova11_supplement": _file_sha256(supplement_path),
    "overlay": _file_sha256(overlay_path),
}
if sha_before != sha_after:
    raise RuntimeError(f"inputs changed during rebuild: {sha_before} -> {sha_after}")

expected = bova_report["inputs"]
checks = {
    "course_sqlite": (sha_after["course_sqlite"], expected["course_sha256"]),
    "bova11_supplement": (sha_after["bova11_supplement"], expected["bova11_sha256"]),
    "overlay": (sha_after["overlay"], expected["overlay_sha256"]),
}
for name, (actual, wanted) in checks.items():
    if actual != wanted:
        raise RuntimeError(f"{name} hash mismatch: {actual} != {wanted}")

audit_path = project_root / "results" / "etapa4_petr4_bova11_audit.json"
bova_audit = json.loads(audit_path.read_text(encoding="utf-8"))
n_qtd = sum(rec["n"] for rec in bova_report["quarterly_accumulation"]["rows"]["groups"]["b_history_clock"])
if not bova_report["verification_ok"]:
    raise RuntimeError("verification_ok is false")
if bova_report["daily_ppi"]["n_scored"] != 126:
    raise RuntimeError("daily n_scored != 126")
if n_qtd != 372:
    raise RuntimeError("quarterly observations != 372")
if not bova_audit.get("accepted", True) and bova_audit.get("issues"):
    raise RuntimeError(f"audit issues: {bova_audit['issues']}")

print("python", platform.python_version(), sys.executable)
for pkg in ("numpy", "pandas", "scikit-learn", "statsmodels"):
    print(pkg, importlib.metadata.version(pkg))
print("freeze", expected["course_sqlite"], expected["course_sha256"])
print("bova11", expected["bova11_file"], expected["bova11_sha256"])
print("overlay", expected["overlay_file"], expected["overlay_sha256"])
print("calendar_n", expected["calendar_n"])
print("verification_ok", bova_report["verification_ok"])
print("daily n", bova_report["daily_ppi"]["n_scored"], "selection", bova_report["daily_ppi"]["selection"])
print("qtd n", n_qtd, "quarters", bova_report["quarterly_accumulation"]["quarters"])
print("qtd selection", bova_report["quarterly_accumulation"]["selection"])
print("audit accepted", bova_audit.get("accepted"), "common", bova_audit.get("common_sessions"), "issues", len(bova_audit.get("issues") or []))


python 3.12.13 /home/ricardo/miniconda3/envs/gas/bin/python
numpy 2.4.3
pandas 2.3.3
scikit-learn 1.8.0
statsmodels 0.14.6
freeze gas_flare.sqlite b314283c8e711becd802d5162be38b1c47052e9603d8bbf81345916fc2bf35ba
bova11 etapa4_bova11_20261018T152955Z.sqlite 637b3ad353aae2fe75a477f4bd96083b9e843a9e781b1d0f727f261335d77281
overlay etapa4_equity_overlay_2023-11-17.json 8348b60445998a682e64bf903d44b62690837f33e42040e9d63a77d156829c75
calendar_n 1157
verification_ok True
daily n 126 selection {'selected': None, 'status': 'unresolved_tie', 'candidates': ['a_ppi_l1', 'a_ppi_l2', 'a_ppi_l3'], 'min_RMSE': 0.01917607279349858, 'min_error_group': 'a_ppi_l2', 'within_2pct': ['a_ppi_l1', 'a_ppi_l2', 'a_ppi_l3'], 'n_feat_selected_class': 4}
qtd n 372 quarters ['2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2']
qtd selection {'selected': 'b_history_clock', 'status': 'simplicity', 'min_RMSE': 0.014604013328020256, 'min_error_group': 'b_history_clock_prod', 'within_2pct': ['b_history_clock', 'b

# **Resultados**

### Referência histórica — Etapa 3

*(As Tabelas 1 a 4 e as Figuras 14 a 27 repetem o recorte da Etapa 3: erro de vários passos, dummy que não se atualiza no horizonte e JSON congelado. Não usam o protocolo de um passo desta etapa.)*

O modelo base desta etapa é a linha ingênua e o SARIMAX com `prod` contemporâneo, no mesmo treino e no mesmo teste de cada vetor. A LSTM e o TimesFM 3.0 entram no vetor diário com `prod`; o Prophet entra no preço e na atividade. As Tabelas 1 a 4 reúnem MAE, MSE e RMSE; o horizonte curto é prefixo do horizonte longo. Os erros saem da passagem M4 com `prod` (LSTM janela 21, TimesFM MLX). A leitura crítica do melhor resultado fica na Discussão da Etapa 4.

A Tabela 1 apresenta MAE, MSE e RMSE do vetor de mercado. No teste (20 de agosto a 18 de setembro de 2026) PETR4 abre a R＄ 44,32, cai a R＄ 41,35 e fecha a R＄ 48,50, enquanto a linha ingênua repete o último fechamento de treino, R＄ 43,11. O SARIMAX (0,1,0) com `prod` do dia fica em RMSE 1,41 R＄ em cinco pregões e 4,21 R＄ em 21, a cerca de R＄ 0,10 da linha ingênua no horizonte longo. A LSTM com `prod` (janela 21) tem MAE 1,25 R＄ em cinco dias e RMSE 5,77 R＄ em 21. O TimesFM com `prod` do treino fica em RMSE 1,38 R＄ e 4,67 R＄. A Figura 14 mostra essa descida e essa subida: as famílias permanecem perto do último fechamento de treino.

Tabela 1 — Mercado (PETR4, R＄)

| Modelo | Horizonte | MAE | MSE | RMSE |
|---|---|---:|---:|---:|
| naive | 5d | 1,36 | 1,95 | 1,40 |
| naive | 21d | 3,74 | 18,61 | 4,31 |
| sarimax (0,1,0) | 5d | 1,38 | 2,00 | 1,41 |
| sarimax (0,1,0) | 21d | 3,66 | 17,72 | 4,21 |
| lstm + `prod` (janela 21) | 5d | 1,25 | 2,19 | 1,48 |
| lstm + `prod` (janela 21) | 21d | 4,94 | 33,28 | 5,77 |
| timesfm + `prod` | 5d | 1,36 | 1,92 | 1,38 |
| timesfm + `prod` | 21d | 4,03 | 21,79 | 4,67 |

![PETR4 real × previsto no teste](../figures/forecasts/fc_mercado_petr4.svg)

*Figura 14 — PETR4 no teste, 20 de agosto a 18 de setembro de 2026, real e previsto.*

A Tabela 2 apresenta MAE e RMSE do vetor de preço. O preço de lista da Petrobras permanece quase constante nas oito semanas de teste, de modo que a linha ingênua fica abaixo de 0,001 R＄/L. Com o `prod` da semana, o SARIMAX (0,1,0) se afasta desse degrau: RMSE 0,045 R＄/L na gasolina e 0,123 R＄/L no diesel em quatro semanas. O Prophet, com sazonalidade anual, afasta-se 0,42 a 0,56 R＄/L. No diesel ANP o SARIMAX reduz o RMSE de 0,55 para 0,45 R＄/L em quatro semanas. Na gasolina ANP o `auto_arima` devolve (1,0,0) sem diferença e o SARIMAX explode (RMSE 3,55 R＄/L em quatro semanas, contra 0,10 da naive), porque o `prod` está em R＄/dia e o preço em R＄/L. As Figuras 15 a 18 mostram o diesel e a gasolina da ANP e os preços de lista da Petrobras no teste.

Tabela 2 — Preço (R＄/L)

| Alvo | Horizonte | MAE naive | RMSE naive | MAE SARIMAX | RMSE SARIMAX | MAE Prophet | RMSE Prophet |
|---|---|---:|---:|---:|---:|---:|---:|
| ANP gasolina | 4w | 0,094 | 0,101 | 3,45 | 3,55 | 0,466 | 0,471 |
| ANP gasolina | 8w | 0,121 | 0,149 | 4,68 | 4,92 | 0,512 | 0,541 |
| ANP diesel | 4w | 0,399 | 0,547 | 0,359 | 0,450 | 0,421 | 0,555 |
| ANP diesel | 8w | 0,230 | 0,391 | 0,255 | 0,337 | 0,760 | 0,873 |
| Petrobras gasolina | 4w | 0,0004 | 0,0004 | 0,041 | 0,045 | 0,418 | 0,419 |
| Petrobras gasolina | 8w | 0,0004 | 0,0004 | 0,053 | 0,056 | 0,417 | 0,418 |
| Petrobras diesel | 4w | 0,0001 | 0,0002 | 0,117 | 0,123 | 0,563 | 0,563 |
| Petrobras diesel | 8w | 0,0002 | 0,0003 | 0,158 | 0,166 | 0,454 | 0,469 |

![ANP diesel real × previsto](../figures/forecasts/fc_preco_anp_die.svg)

*Figura 15 — Preço ANP do diesel no teste, real e previsto.*

![ANP gasolina real × previsto](../figures/forecasts/fc_preco_anp_gas.svg)

*Figura 16 — Preço ANP da gasolina no teste, real e previsto.*

![Petrobras gasolina real × previsto](../figures/forecasts/fc_preco_pbr_gas.svg)

*Figura 17 — Preço de lista Petrobras da gasolina no teste, real e previsto.*

![Petrobras diesel real × previsto](../figures/forecasts/fc_preco_pbr_die.svg)

*Figura 18 — Preço de lista Petrobras do diesel no teste, real e previsto.*

A Tabela 3 apresenta as métricas do vetor de atividade. No painel mensal o `prod` do mês e o termo sazonal reduzem o erro da linha ingênua no tráfego, na indústria e no varejo. No tráfego pesado o RMSE de três meses cai de 10,31 (naive) para 8,89 (SARIMAX) e 4,37 (Prophet), e na indústria de 6,77 para 2,38 e 1,69. No varejo o Prophet fica em RMSE 0,88 no horizonte de três meses, contra 2,72 da linha ingênua, e no desemprego o SARIMAX (0,1,2)(1,0,0)[12] tem RMSE 0,19 p.p. em três meses. As Figuras 19 a 22 mostram a série vista em preto e, a partir de abril de 2026, a naive, o SARIMAX e o Prophet contra mai., jun. e jul. 2026: tráfego pesado, indústria, varejo e desemprego.

Tabela 3 — Atividade

| Alvo | Horizonte | MAE naive | RMSE naive | MAE SARIMAX | RMSE SARIMAX | MAE Prophet | RMSE Prophet |
|---|---|---:|---:|---:|---:|---:|---:|
| tráfego pesado | 1m | 15,30 | 15,30 | 3,68 | 3,68 | 2,34 | 2,34 |
| tráfego pesado | 3m | 9,43 | 10,31 | 6,54 | 8,89 | 3,57 | 4,37 |
| indústria | 1m | 5,35 | 5,35 | 2,24 | 2,24 | 2,07 | 2,07 |
| indústria | 3m | 6,26 | 6,77 | 2,12 | 2,38 | 1,67 | 1,69 |
| varejo | 1m | 3,67 | 3,67 | 3,50 | 3,50 | 0,61 | 0,61 |
| varejo | 3m | 2,28 | 2,72 | 2,11 | 2,48 | 0,84 | 0,88 |
| desemprego | 1m | 0,20 | 0,20 | 0,13 | 0,13 | 0,29 | 0,29 |
| desemprego | 3m | 0,37 | 0,39 | 0,19 | 0,19 | 0,39 | 0,40 |

![Tráfego pesado real × previsto](../figures/forecasts/fc_atividade_abcr_heavy.svg)

*Figura 19 — Tráfego pesado ABCR, série vista e previsão (mai.–jul. 2026).*

![Indústria real × previsto](../figures/forecasts/fc_atividade_pim.svg)

*Figura 20 — Produção industrial (PIM-PF), série vista e previsão (mai.–jul. 2026).*

![Varejo real × previsto](../figures/forecasts/fc_atividade_pmc.svg)

*Figura 21 — Volume de vendas no varejo (PMC), série vista e previsão (mai.–jul. 2026).*

![Desemprego real × previsto](../figures/forecasts/fc_atividade_unemp.svg)

*Figura 22 — Taxa de desemprego (PNAD), série vista e previsão (mai.–jul. 2026).*

A Tabela 4 apresenta o 2T26 publicado contra o último trimestre de treino (1T26). A receita RT&M publicada no 2T26 é R＄ 163.193 milhões, R＄ 46.015 milhões acima do 1T26, e a margem cai de 20,27% para 16,01%, enquanto o volume de diesel sobe de 738 para 742 mbpd e o de gasolina cai de 413 para 404 mbpd. A nota 4 do ITR registra R＄ 9.737 milhões de subvenção ao diesel e R＄ 816 milhões à gasolina, fato sem paralelo no treino. O SARIMAX recebe o produto trimestral `prod_pbr` (desvio do PPI vezes o volume Petrobras do próprio trimestre) e diferencia com `d=1`. No 2T26 essa exógena soma abril a junho de 2026. Com o regressor, a receita prevista sobe de R＄ 117.178 milhões (naive) para R＄ 129.165 milhões, ainda R＄ 34.028 milhões abaixo do publicado. Diesel, lucro bruto e margem ficam mais longe da linha ingênua: o modelo empurra a margem para 23,99% enquanto o ITR cai a 16,01%. Com um ponto de teste e a subvenção, a Tabela 4 permanece indicação. As Figuras 23 a 27 apresentam as projeções do último trimestre (2T26) sobre a série vista em preto: diesel, gasolina, receita, lucro bruto e margem, com a naive e o SARIMAX a partir do 1T26.

Tabela 4 — Companhia, 2T26 publicado contra o último trimestre de treino (1T26)

| Alvo | 1T26 (naive) | SARIMAX | 2T26 publicado | MAE naive | MAE SARIMAX | ordem |
|---|---:|---:|---:|---:|---:|---|
| diesel mbpd | 738 | 785 | 742 | 4 | 42,91 | (2,1,2) |
| gasolina mbpd | 413 | 395 | 404 | 9 | 9,11 | (0,1,2) |
| receita RT&M (R＄ milhões) | 117.178 | 129.165 | 163.193 | 46.015 | 34.028 | (0,1,2) |
| lucro bruto RT&M (R＄ milhões) | 23.750 | 28.586 | 26.130 | 2.380 | 2.456 | (0,1,2) |
| margem RT&M (%) | 20,27 | 23,99 | 16,01 | 4,26 | 7,98 | (0,1,2) |

![Diesel mbpd real × previsto](../figures/forecasts/fc_companhia_diesel_mbpd.svg)

*Figura 23 — Diesel Petrobras (mbpd), série vista e previsão no 2T26.*

![Gasolina mbpd real × previsto](../figures/forecasts/fc_companhia_gasoline_mbpd.svg)

*Figura 24 — Gasolina Petrobras (mbpd), série vista e previsão no 2T26.*

![Receita RT&M real × previsto](../figures/forecasts/fc_companhia_rtm_sales_revenue_rsmn.svg)

*Figura 25 — Receita RT&M, série vista e previsão no 2T26.*

![Lucro bruto RT&M real × previsto](../figures/forecasts/fc_companhia_rtm_gross_profit_rsmn.svg)

*Figura 26 — Lucro bruto RT&M, série vista e previsão no 2T26.*

![Margem RT&M real × previsto](../figures/forecasts/fc_companhia_rtm_gross_margin_pct.svg)

*Figura 27 — Margem RT&M, série vista e previsão no 2T26.*


## Resultados recalculados da Etapa 4

A tabela por alvo separa dois números que não são o mesmo julgamento: a menor perda observada e a extensão escolhida depois do desempate de 2%, que fica com a especificação mais simples quando o RMSE está perto do mínimo. O critério de contribuição compara a exposição com o controle correspondente (história só do alvo, história com sazonalidade, ou o mesmo contexto sem a exposição), não apenas com a dummy. As seis comparações da revisão aparecem à parte; os resultados da Etapa 3 acima permanecem históricos.


In [10]:
display(target_table().round(5))
display(review_table().round(5))
display(Markdown(interpretation()))


,vetor,alvo,bloco,experimento,menor erro observado,extensão após desempate,controle correspondente,RMSE extensão,RMSE controle,vitórias vs controle,critério PPI × volume,critério contexto
0,mercado,PETR4,21d,prod_contrib,y_fuels,y_prod,y_only,0.73056,0.73537,4/6,não,não
1,mercado,PETR4 dlog,21d,petr4_vol,y_dqty,y_dprod,y_only,0.01659,0.01654,2/6,não,não
2,mercado,RV 5 pregões,21d,petr4_vol,y_dprod_dqty,y_dprod_dqty,y_dqty,0.00402,0.00405,3/6,não,não
3,mercado,RV 21 pregões,21d,petr4_vol,y_dprod,y_dprod,y_only,0.00088,0.00089,1/6,não,não
4,preço,ANP gasolina,8w,prod_contrib,y_prod,y_prod,y_only,0.38304,0.38701,2/4,não,não
5,preço,ANP diesel,8w,independents,y_pbr0,y_pbr0,y_only,0.28933,0.33745,3/4,sim,não
6,preço,Petrobras gasolina,8w,prod_contrib,dummy,y_dprod,y_only,0.04395,0.02754,2/4,não,não
7,preço,Petrobras diesel,8w,prod_contrib,dummy,y_dprod,y_only,0.10298,0.07015,0/4,não,não
8,preço,ANP gasolina Δ,8w,price_transmission,y_exp_l08,y_exp_l04,y_only,0.38261,0.41066,3/4,não,não
9,preço,ANP diesel Δ,8w,price_transmission,y_list_exp,y_exp0,y_only,0.26172,0.27685,2/4,não,não


,alvo,extensão,controle,RMSE controle,RMSE extensão,MAE controle,MAE extensão,ganho RMSE %,vitórias,critério exploratório
0,ANP diesel,y_fuels,y_only,0.33790,0.28146,0.22354,0.17953,16.70457,1/4,False
1,ANP gasolina,y_ind0,y_only,0.39335,0.36757,0.23397,0.24856,6.55416,3/4,False
2,ANP gasolina Δ,y_exp_l08,y_only,0.41066,0.37727,0.24655,0.26254,8.13022,3/4,False
3,varejo,y_seasonal_exp,y_seasonal,1.90714,1.56240,1.71382,1.27196,18.07622,2/4,False
4,IGP-M mom,y_die_l03,y_only,0.80639,0.85012,0.73819,0.74381,-5.42361,3/4,False
5,IPCA mom,y_anp_gas,y_only,0.25050,0.19945,0.23005,0.18352,20.37719,4/4,True


### Leitura dos resultados recalculados

Com gasolina e diesel em separado (`y_fuels`), o RMSE mediano do diesel ANP cai de 0,3379 para 0,2815 (16,7%), mas o RMSE é menor em 1 de 4 blocos de datas: a mediana cai porque um bloco de 2026 concentra o ganho, e os outros três não melhoram. A exposição Petrobras contemporânea (`y_pbr0`) tem RMSE menor em 3 de 4 blocos contra a história só do alvo; as duas extensões não são o mesmo regressor. O critério pedia também a maioria dos blocos e o MAE mediano que não aumentasse, e por isso `y_fuels` não completa a contribuição incremental. Na gasolina ANP, a exposição dos polos independentes e as defasagens 0–8 da variação também reduzem o RMSE mediano, com RMSE menor em 3 de 4 blocos; o MAE, porém, sobe de 0,2340 para 0,2486 na exposição independente e de 0,2465 para 0,2625 na variação, e o critério falha. No varejo, a exposição entra sobre o controle sazonal `y_seasonal` e o RMSE é menor em 2 de 4 blocos, sem a maioria pedida.

O Índice Geral de Preços–Mercado (IGP-M) com exposição de diesel aumenta o RMSE mediano, de 0,8064 para 0,8501. No Índice Nacional de Preços ao Consumidor Amplo (IPCA), a variação do preço ANP da gasolina reduz o RMSE mediano de 0,2505 para 0,1995 e o RMSE é menor em 4 de 4 blocos, com MAE menor. Esse regressor é a variação do preço ANP de produtores e importadores, não o desvio de paridade ponderado pelo volume. Os regressores da mesma data e as defasagens 0–8 não comprovam antecipação, atraso causal nem viabilidade ex ante, prever só com o que já teria sido publicado. A escolha entre especificações nos mesmos blocos permanece exploratória.

Na grade financeira, o grupo escolheu o método com 2019–2021 e não o trocou depois (a trava). A receita permanece com persistência e história completa. O lucro fica com SARIMA(0,1,0)×(1,0,0)₄ e reduz o RMSE agrupado, a raiz sobre os trimestres juntos, do 1º trimestre de 2022 ao 1º trimestre de 2026 em 0,70% frente à persistência, abaixo de 5%. A margem fica com SARIMA(0,0,0)×(0,0,0)₄, um nível médio da série; a persistência tem RMSE agrupado menor na avaliação (3,5233 contra 4,5583). Nenhum dos três atende o diagnóstico trimestral contra a persistência. Esse diagnóstico usa RMSE agrupado, MAE médio e origens com menor erro absoluto; não é a regra da mediana por bloco dos ensaios diários, semanais e mensais.

Com exposição de diesel, a correção do resíduo do 1º trimestre de 2024 ao 1º trimestre de 2026 aumenta o RMSE da receita e reduz os do lucro e da margem. A comparação com 1T24–4T25 retira somente o 1º trimestre de 2026 (nove origens contra oito): as três correções aumentam o RMSE (receita 4,9%, lucro 9,6%, margem 87,4%), de modo que o ganho de lucro e margem nessa janela depende da inclusão do primeiro trimestre de 2026. O 2º trimestre de 2026 entra à parte, mistura ganhos e perdas e não estabelece estabilidade. Receita e lucro entram em R＄ milhões; o RMSE da margem, em pontos percentuais. O ensaio residual descreve a associação contemporânea e não promove um modelo de previsão.


A Figura 28 compara o RMSE de cada extensão com o respectivo controle nos blocos de avaliação. Cada painel usa a unidade do próprio alvo; a altura das curvas não deve ser comparada entre painéis.


![Figura 28 — RMSE por bloco](../figures/etapa4/erros_por_bloco.svg)


*Figura 28 — RMSE por bloco nas seis comparações selecionadas da Etapa 4. Fonte: elaborado pelos autores a partir dos resultados da Etapa 4.*


### Síntese da contribuição incremental

A Figura 29 reúne a redução ou o aumento percentual do RMSE mediano, as vitórias por bloco e a mudança do MAE mediano, sempre frente ao controle correspondente. As três medidas devem ser lidas juntas: reduzir o RMSE mediano em pelo menos 5% não basta para atender ao critério exploratório. O resultado do IPCA usa preço ANP e aparece separado das exposições PPI × volume.


![Figura 29 — Contribuição incremental](../figures/etapa4/contribuicao_incremental.svg)


*Figura 29 — Contribuição incremental nas comparações selecionadas. A barra à direita indica RMSE mediano menor que o do controle identificado na linha. As anotações distinguem redução e aumento do RMSE e do MAE, calculados contra esse controle. Seis comparações, com 32 observações semanais (dezembro de 2025 a julho de 2026) ou 12 mensais (maio de 2025 a abril de 2026). Fonte: elaborado pelos autores a partir dos resultados da Etapa 4.*


### Preços e inflação: observado e estimado

A Figura 30 mostra o preço observado do diesel ANP, a estimativa baseada na história do alvo e a estimativa com exposição Petrobras contemporânea (`y_pbr0`). As faixas identificam os blocos de avaliação. As Figuras 30 e 31 mostram quando as diferenças entre os modelos aparecem ao longo da avaliação. A comparação por bloco é apresentada na Figura 32.


![Figura 30 — Diesel ANP observado e estimativas](../figures/etapa4/temporal_diesel_anp.svg)


*Figura 30 — Diesel ANP observado e estimativas de `y_only` e `y_pbr0`, em R＄/L. Quatro blocos semanais de 8 de dezembro de 2025 a 13 de julho de 2026 (32 observações). As faixas indicam os blocos de avaliação. O modelo é ajustado no início de cada bloco, e seus coeficientes permanecem fixos dentro dele. As estimativas de um passo usam valores anteriores observados do alvo e regressores contemporâneos realizados. As linhas ligam datas consecutivas; a viabilidade ex ante não foi avaliada. Erro = observado − estimado: positivo, o modelo subestima o observado; negativo, o modelo superestima. Fonte: elaborado pelos autores a partir dos resultados da Etapa 4.*


A Figura 31 apresenta o IPCA observado, o controle baseado na história do alvo e a extensão com variação do preço ANP da gasolina. Essa série de preços corresponde a produtores e importadores. O gráfico mostra uma associação contemporânea, sem demonstrar antecipação da inflação.


![Figura 31 — IPCA observado e estimativas](../figures/etapa4/temporal_ipca.svg)


*Figura 31 — IPCA mensal observado e estimativas de `y_only` e `y_anp_gas`, em %. Quatro blocos mensais de maio de 2025 a abril de 2026 (12 observações). As faixas indicam os blocos de avaliação. O modelo é ajustado no início de cada bloco, e seus coeficientes permanecem fixos dentro dele. As estimativas de um passo usam valores anteriores observados do alvo e regressores contemporâneos realizados. As linhas ligam datas consecutivas; a viabilidade ex ante não foi avaliada. Erro = observado − estimado: positivo, o modelo subestima o observado; negativo, o modelo superestima. O regressor de contexto é preço ANP, não PPI × volume. Fonte: elaborado pelos autores a partir dos resultados da Etapa 4.*


As trajetórias das Figuras 30 e 31 são complementadas pela Figura 32, que mostra a redução do RMSE em cada bloco. As duas especificações de diesel usam regressores distintos: `y_fuels` separa as exposições por combustível, enquanto `y_pbr0` usa uma exposição contemporânea. A comparação preserva o controle de cada experimento.


![Figura 32 — Redução do RMSE por bloco](../figures/etapa4/estabilidade_por_bloco.svg)


*Figura 32 — Redução percentual do RMSE por bloco para duas especificações de diesel ANP e uma extensão de IPCA. Valores positivos indicam menor erro que o controle. Os períodos semanais e mensais são identificados separadamente. Fonte: elaborado pelos autores a partir dos resultados da Etapa 4.*


### Demanda e mobilidade: sazonalidade e exposição

A Figura 33 separa a contribuição da sazonalidade da contribuição adicional da exposição. Para cada alvo, o RMSE mediano do modelo baseado apenas na história recebe índice 100. As outras barras representam o controle com defasagem 12 e esse mesmo controle acrescido de exposição.


![Figura 33 — RMSE mediano relativo à história do alvo](../figures/etapa4/sazonalidade_e_exposicao.svg)


*Figura 33 — RMSE mediano relativo à história do alvo, definida como 100. Valores menores indicam menor erro. Os índices permitem comparar reduções relativas, não os erros absolutos de séries com unidades diferentes. Quatro alvos, blocos mensais de três meses. Fonte: elaborado pelos autores a partir dos resultados da Etapa 4.*


A Figura 34 detalha a evolução temporal do varejo, com a série observada, o controle sazonal e a extensão com exposição. A aproximação entre estimado e observado varia entre os blocos; o gráfico deve ser interpretado em conjunto com as métricas agregadas e a comparação por bloco.


![Figura 34 — Varejo observado e estimativas](../figures/etapa4/temporal_varejo.svg)


*Figura 34 — Varejo observado e estimativas de `y_seasonal` e `y_seasonal_exp`, índice mensal do Volume de vendas no varejo (PMC). Quatro blocos mensais de maio de 2025 a abril de 2026 (12 observações). As faixas indicam os blocos de avaliação. O modelo é ajustado no início de cada bloco, e seus coeficientes permanecem fixos dentro dele. As estimativas de um passo usam valores anteriores observados do alvo e regressores contemporâneos realizados. As linhas ligam datas consecutivas; a viabilidade ex ante não foi avaliada. Erro = observado − estimado: positivo, o modelo subestima o observado; negativo, o modelo superestima. Fonte: elaborado pelos autores a partir dos resultados da Etapa 4.*


### História financeira e correção do resíduo

A tabela compacta traz o método e a janela que o grupo escolheu com os trimestres de 2019 a 2021 e não trocou depois (a trava), o RMSE agrupado dessa seleção e o da avaliação do 1º trimestre de 2022 ao 1º trimestre de 2026, e a persistência como referência explícita. O RMSE agrupado junta todos os trimestres da janela numa só raiz; não é a mediana por bloco usada nos ensaios diários, semanais e mensais. A margem escolhida estima um nível médio; o nome SARIMA(0,0,0)×(0,0,0)₄ não descreve um ciclo sazonal. O ranking posterior a 2021 aparece no caderno auxiliar e não escolhe outro método.

A segunda tabela compara a correção do resíduo, o valor realizado menos essa previsão-base, quando se acrescenta a exposição de diesel, nas mesmas datas elegíveis. Os três períodos (1º trimestre de 2024 a 1º trimestre de 2026, o mesmo recorte sem o 1º trimestre de 2026, e o ponto legado do 2º trimestre de 2026) entram em linhas separadas, porque um ganho que some sem o 1T26 não se lê como estabilidade. Receita e lucro estão em R＄ milhões. Nas tabelas, o RMSE da margem está em pontos percentuais; a Figura 35 mostra o nível da margem em %. A grade completa e as exposições de gasolina e de `prod` combinado ficam no caderno auxiliar.


In [11]:
display(finance_locked_table().round(4))
display(finance_sensitivity_official_table().round(4))


,alvo,método,janela de história,RMSE seleção 1T19–4T21,RMSE avaliação 1T22–1T26,RMSE persistência 1T22–1T26,ganho RMSE % vs persistência,diagnóstico 5% vs persistência,n avaliação
0,receita RT&M,persistência,história completa,12929.6344,12393.9153,12393.9153,0.0000,não atendido,17
1,lucro bruto RT&M,"SARIMA(0,1,0)×(1,0,0)₄",história completa,3516.4626,5070.0939,5105.6093,0.6956,não atendido,17
2,margem RT&M,"SARIMA(0,0,0)×(0,0,0)₄",história completa,4.7886,4.5583,3.5233,-29.3738,não atendido,17


,alvo,método,janela de história,exposição,período,n,RMSE controle,RMSE correção,ganho RMSE %
0,receita RT&M,persistência,história completa,diesel,1T24–1T26,9,7281.2165,12075.2352,-65.8409
1,receita RT&M,persistência,história completa,diesel,1T24–4T25,8,7642.9957,8018.1032,-4.9079
2,receita RT&M,persistência,história completa,diesel,2T26 (legado),1,46015.0000,17135.0529,62.7620
3,lucro bruto RT&M,"SARIMA(0,1,0)×(1,0,0)₄",história completa,diesel,1T24–1T26,9,4245.5825,2694.5103,36.5338
4,lucro bruto RT&M,"SARIMA(0,1,0)×(1,0,0)₄",história completa,diesel,1T24–4T25,8,2576.8217,2823.7179,-9.5814
5,lucro bruto RT&M,"SARIMA(0,1,0)×(1,0,0)₄",história completa,diesel,2T26 (legado),1,2330.2937,16463.5755,-606.5022
6,margem RT&M,"SARIMA(0,0,0)×(0,0,0)₄",história completa,diesel,1T24–1T26,9,4.5566,4.1383,9.1815
7,margem RT&M,"SARIMA(0,0,0)×(0,0,0)₄",história completa,diesel,1T24–4T25,8,1.7764,3.3293,-87.4242
8,margem RT&M,"SARIMA(0,0,0)×(0,0,0)₄",história completa,diesel,2T26 (legado),1,8.2976,4.6064,44.4853


A Figura 35 mostra o lucro bruto e a margem de RT&M observados, a previsão-base do método escolhido em 2019–2021 e a correção com exposição de diesel. O destaque do 1T26 e a separação do ponto legado 2T26 permitem examinar a influência desses trimestres sem reunir as duas avaliações. A avaliação principal da correção abrange 1T24–1T26, com nove origens. O 2T26 é apresentado separadamente e não integra essa comparação.


![Figura 35 — Lucro bruto e margem de RT&M](../figures/etapa4/correcao_financeira_no_tempo.svg)


*Figura 35 — Lucro bruto e margem de RT&M: observado, baseline travado e correção com diesel, de 1T24 a 2T26. Lucro em R＄ milhões; margem em %. Dez trimestres. A avaliação principal da correção abrange 1T24–1T26, com nove origens. O 2T26 é apresentado separadamente e não integra essa comparação. A correção usa exposição contemporânea e permanece descritiva. Fonte: elaborado pelos autores a partir dos resultados da Etapa 4.*


A Figura 36 quantifica a sensibilidade observada na Figura 35. Compara a redução do RMSE de receita, lucro e margem em 1T24–1T26, com nove origens, e em 1T24–4T25, com oito. A diferença entre os recortes é somente a retirada do 1T26; o 2T26 não integra essas barras.


![Figura 36 — Sensibilidade da correção financeira ao 1T26](../figures/etapa4/sensibilidade_1t26.svg)


*Figura 36 — Redução percentual do RMSE pela correção com diesel, frente ao baseline travado. A análise de sensibilidade compara 1T24–1T26, com nove origens, e 1T24–4T25, com oito. A segunda janela retira somente o 1T26. O 2T26 permanece separado. Valores positivos indicam redução do erro; valores negativos indicam aumento. Fonte: elaborado pelos autores a partir dos resultados da Etapa 4.*


### PETR4 em relação ao mercado: duas dimensões de retorno

O ensaio pergunta se o PPI, em nível ou acumulado com volume, acompanha o retorno da PETR4 depois que se subtrai o BOVA11. Esse alvo relativo ao mercado é distinto dos ensaios históricos de nível de preço e de `dlog`, a variação logarítmica do fechamento. A Tabela 5 e a Figura 37 cobrem o diferencial diário com PPI defasado; a Tabela 6 e as Figuras 38 e 39 cobrem o diferencial acumulado no trimestre com exposição PPI × volume. Os dois recortes não se misturam: 126 pregões de retorno diário não são os 372 pontos dos seis caminhos trimestrais.


**Tabela 5 — Diferencial diário PETR4 − BOVA11: referências e contribuição incremental do PPI.**

As métricas são medianas nos seis blocos emparelhados de 21 pregões (19 de fevereiro a 19 de agosto de 2026; 126 observações). Redução percentual positiva significa RMSE menor do que o do controle nomeado. Zero e persistência atualizada entram como referências; a linha `a_history` compara-se com o zero. Cada candidato de PPI compara-se com `a_history`.


In [12]:
A = bova_report["daily_ppi"]
DASH = "—"

def _pp(value):
    return 100.0 * float(value)

def _pct(value):
    return 100.0 * float(value)

def _wins(node):
    return f"{node['blocks_better_RMSE']}/{node['n_blocks']}"

def _crit(flag):
    return "sim" if flag else "não"

def _ref_row(name, node, control):
    if control is None:
        return {
            "especificação": name,
            "controle emparelhado": DASH,
            "RMSE mediano (p.p.)": _pp(node["median_RMSE"]),
            "MAE mediano (p.p.)": _pp(node["median_MAE"]),
            "redução RMSE (%)": DASH,
            "vitórias/6": DASH,
            "critério incremental": DASH,
        }
    return {
        "especificação": name,
        "controle emparelhado": control,
        "RMSE mediano (p.p.)": _pp(node["median_RMSE"]),
        "MAE mediano (p.p.)": _pp(node["median_MAE"]),
        "redução RMSE (%)": _pct(node["rel_RMSE_vs_pers"]),
        "vitórias/6": _wins(node),
        "critério incremental": _crit(node["promote_ge_5pct"]),
    }

rows = [
    _ref_row("zero", A["references"]["zero"], None),
    _ref_row("persistência", A["references"]["persistence"], None),
    _ref_row("a_history", A["references"]["a_history_vs_zero"], "zero"),
]
for name in ("a_ppi_l1", "a_ppi_l2", "a_ppi_l3", "a_ppi_l4", "a_ppi_l5"):
    rows.append(_ref_row(name, A["summary"][name], "a_history"))
print("Tabela 5 — medianas nos seis blocos de 21 pregões")
display(
    pd.DataFrame(rows).style.format({
        "RMSE mediano (p.p.)": "{:.3f}",
        "MAE mediano (p.p.)": "{:.3f}",
        "redução RMSE (%)": lambda value: (
            value if isinstance(value, str) else f"{value:.2f}"
        ),
    })
)
print("seleção", A["selection"])
print("a_history vs persistência: redução RMSE %", round(_pct(A["references"]["a_history_vs_persistence"]["rel_RMSE_vs_pers"]), 2),
      "vitórias", _wins(A["references"]["a_history_vs_persistence"]))


Tabela 5 — medianas nos seis blocos de 21 pregões


,especificação,controle emparelhado,RMSE mediano (p.p.),MAE mediano (p.p.),redução RMSE (%),vitórias/6,critério incremental
0,zero,—,2.011,1.537,—,—,—
1,persistência,—,2.701,2.122,—,—,—
2,a_history,zero,2.012,1.543,-0.05,2/6,não
3,a_ppi_l1,a_history,1.925,1.468,4.34,4/6,não
4,a_ppi_l2,a_history,1.918,1.464,4.71,4/6,não
5,a_ppi_l3,a_history,1.938,1.481,3.69,4/6,não
6,a_ppi_l4,a_history,1.965,1.506,2.38,4/6,não
7,a_ppi_l5,a_history,1.988,1.524,1.22,4/6,não


seleção {'selected': None, 'status': 'unresolved_tie', 'candidates': ['a_ppi_l1', 'a_ppi_l2', 'a_ppi_l3'], 'min_RMSE': 0.01917607279349858, 'min_error_group': 'a_ppi_l2', 'within_2pct': ['a_ppi_l1', 'a_ppi_l2', 'a_ppi_l3'], 'n_feat_selected_class': 4}
a_history vs persistência: redução RMSE % 25.5 vitórias 6/6


Nenhuma das cinco defasagens do PPI atende ao critério incremental. A menor mediana entre os candidatos é a de `a_ppi_l2` (cerca de 1,918 p.p. contra 2,012 p.p. de `a_history`; redução de 4,71%; 4/6 blocos), ainda abaixo de 5%. As defasagens 1 a 3 ficam a até 2% dessa menor mediana e têm a mesma complexidade, de modo que o empate permanece sem resolução. `a_history` não supera o zero no agregado (2,012 p.p. contra 2,011 p.p.; 2/6 blocos); a comparação com a persistência, à parte da Tabela 5, mostra redução de 25,5% em 6/6 blocos. No treino até 18 de fevereiro de 2026 (1.005 observações), o maior módulo de correlação entre PPI defasado e o diferencial é cerca de 0,063 (diesel, defasagem 2); o diagnóstico completo fica no caderno auxiliar. O RMSE não informa acerto de direção. A Figura 37 ilustra a defasagem 1, fixada no plano, e não o candidato de menor erro.


![Figura 37 — Diferencial diário PETR4 − BOVA11](../figures/etapa4/petr4_bova11_daily_temporal.svg)


*Figura 37 — Diferencial diário de retorno PETR4 − BOVA11 (`ret_diff_petr4_bova11`; E4-BOVA-A2) nos seis blocos de 21 pregões, de 19 de fevereiro a 19 de agosto de 2026 (126 sessões), em pontos percentuais. Linhas: observado, zero, controle `a_history` e ilustração fixa `a_ppi_l1`. A sombreação marca os blocos de modelo. As linhas ligam pregões consecutivos válidos. Painel inferior: erro = observado − estimado. Informação: um passo, coeficientes fixos dentro do bloco, defasagens observadas; o PPI entra em pregões anteriores. A ilustração de defasagem 1 não é o candidato de menor erro. Fonte: elaborado pelos autores a partir dos resultados da Etapa 4.*


**Tabela 6 — Diferencial acumulado PETR4 − BOVA11: controles e exposição com reset trimestral.**

As métricas são medianas nos seis trimestres civis completos 2025Q1–2026Q2 (372 observações). `b_clock` compara-se com a persistência com reset; `b_clock_prod` e `b_history_clock` comparam-se com `b_clock`; as duas extensões de exposição comparam-se com `b_history_clock`. O critério é específico dessa comparação: um resultado da história contra o relógio não evidencia o PPI × volume.


In [13]:
B = bova_report["quarterly_accumulation"]
DASH = "—"

def _pp(value):
    return 100.0 * float(value)

def _pct(value):
    return 100.0 * float(value)

def _wins(node):
    return f"{node['blocks_better_RMSE']}/{node['n_blocks']}"

def _crit(flag):
    return "sim" if flag else "não"

def _ref_row(name, node, control):
    if control is None:
        return {
            "especificação": name,
            "controle emparelhado": DASH,
            "RMSE mediano (p.p.)": _pp(node["median_RMSE"]),
            "MAE mediano (p.p.)": _pp(node["median_MAE"]),
            "redução RMSE (%)": DASH,
            "vitórias/6": DASH,
            "critério incremental": DASH,
        }
    return {
        "especificação": name,
        "controle emparelhado": control,
        "RMSE mediano (p.p.)": _pp(node["median_RMSE"]),
        "MAE mediano (p.p.)": _pp(node["median_MAE"]),
        "redução RMSE (%)": _pct(node["rel_RMSE_vs_pers"]),
        "vitórias/6": _wins(node),
        "critério incremental": _crit(node["promote_ge_5pct"]),
    }

ctrl = {
    "b_clock": ("persistência com reset", B["references"]["b_clock_vs_persistence"]),
    "b_clock_prod": ("b_clock", B["summary"]["b_clock_prod"]),
    "b_history_clock": ("b_clock", B["summary"]["b_history_clock"]),
    "b_history_clock_prod": ("b_history_clock", B["summary"]["b_history_clock_prod"]),
    "b_history_clock_fuels": ("b_history_clock", B["summary"]["b_history_clock_fuels"]),
}
rows = [
    _ref_row("zero", B["references"]["zero"], None),
    _ref_row("persistência com reset", B["references"]["persistence"], None),
]
for name, (control, node) in ctrl.items():
    rows.append(_ref_row(name, node, control))
print("Tabela 6 — medianas nos seis trimestres 2025Q1–2026Q2")
display(
    pd.DataFrame(rows).style.format({
        "RMSE mediano (p.p.)": "{:.3f}",
        "MAE mediano (p.p.)": "{:.3f}",
        "redução RMSE (%)": lambda value: (
            value if isinstance(value, str) else f"{value:.2f}"
        ),
    })
)
hp = B["references"]["b_history_clock_vs_persistence"]
print("b_history_clock vs persistência com reset: RMSE", round(_pp(hp["median_RMSE"]), 3),
      "ctrl", round(_pp(hp["persistence_median_RMSE"]), 3),
      "redução %", round(_pct(hp["rel_RMSE_vs_pers"]), 2),
      "vitórias", _wins(hp), "critério", _crit(hp["promote_ge_5pct"]))
print("seleção", B["selection"])


Tabela 6 — medianas nos seis trimestres 2025Q1–2026Q2


,especificação,controle emparelhado,RMSE mediano (p.p.),MAE mediano (p.p.),redução RMSE (%),vitórias/6,critério incremental
0,zero,—,6.648,5.585,—,—,—
1,persistência com reset,—,1.370,1.036,—,—,—
2,b_clock,persistência com reset,7.072,6.335,-416.08,0/6,não
3,b_clock_prod,b_clock,13.669,10.465,-93.28,4/6,não
4,b_history_clock,b_clock,1.464,1.119,79.30,6/6,sim
5,b_history_clock_prod,b_history_clock,1.460,1.116,0.23,3/6,não
6,b_history_clock_fuels,b_history_clock,1.470,1.131,-0.41,2/6,não


b_history_clock vs persistência com reset: RMSE 1.464 ctrl 1.37 redução % -6.82 vitórias 1/6 critério não
seleção {'selected': 'b_history_clock', 'status': 'simplicity', 'min_RMSE': 0.014604013328020256, 'min_error_group': 'b_history_clock_prod', 'within_2pct': ['b_history_clock', 'b_history_clock_prod', 'b_history_clock_fuels'], 'n_feat_selected_class': 4}


A exposição total (`b_history_clock_prod`) tem o menor RMSE mediano entre as especificações ajustadas (cerca de 1,460 p.p.; redução de 0,23% contra `b_history_clock`; 3/6 trimestres) e não atende ao critério incremental. Os combustíveis em separado pioram um pouco a mediana. A regra de simplicidade retém `b_history_clock`. O ganho de cerca de 79,3% da história contra o relógio aparece na Tabela 6 ao lado do resultado mais fraco contra a persistência com reset: RMSE 1,464 p.p. contra 1,370 p.p., com vitória em só 1/6 trimestres. O relógio sozinho perde para a persistência (cerca de 7,07 p.p. contra 1,37 p.p.). Os diagnósticos por trimestre e as tabelas de erro terminal permanecem no caderno auxiliar; a instabilidade dentro do trimestre não conta como réplica independente. A Figura 38 mostra os caminhos descritivos do diferencial e da exposição, em unidades diferentes.


![Figura 38 — Diferencial acumulado e exposição por trimestre](../figures/etapa4/petr4_bova11_qtd_paths.svg)


*Figura 38 — Diferencial composto PETR4 − BOVA11 (`ret_diff_qtd_petr4_bova11`) e exposição acumulada PPI × volume (`C_total_qtd`) nos seis trimestres completos 2025Q1–2026Q2 (E4-BOVA-B1; 372 observações). Três linhas de pares de painéis, em duas colunas. Unidades: pontos percentuais no retorno e R＄ milhões na exposição; as escalas dos painéis diferem. Amostra: pregões de cada trimestre, com base no último pregão anterior. Informação: exposição contemporânea, descritiva. Semelhança visual não é teste de correlação. Fonte: elaborado pelos autores a partir dos resultados da Etapa 4.*


A Figura 38 descreve os dois acumuladores no recorte; a Figura 39 passa das trajetórias à estimativa de um passo e ao erro, com persistência de reset, controle de história e a ilustração de exposição total.


![Figura 39 — Diferencial acumulado no trimestre: observado e estimativas](../figures/etapa4/petr4_bova11_qtd_temporal.svg)


*Figura 39 — Diferencial acumulado no trimestre PETR4 − BOVA11 (`ret_diff_qtd_petr4_bova11`; E4-BOVA-B2) de 2025Q1 a 2026Q2 (372 observações), em pontos percentuais, com persistência de reset, controle `b_history_clock` e ilustração `b_history_clock_prod`. Painel inferior: erro = observado − estimado. As interrupções nas fronteiras são resets deliberados; dentro do trimestre as linhas ligam pregões válidos. Informação: um passo, coeficientes fixos por trimestre, defasagens observadas do alvo, exposição contemporânea. O ajuste próximo reflete sobretudo a persistência, não uma previsão de horizonte longo bem-sucedida. A ilustração de exposição total também tem o menor RMSE entre as especificações ajustadas nesta execução; o ganho não atende ao critério incremental, e a simplicidade retém `b_history_clock`. Fonte: elaborado pelos autores a partir dos resultados da Etapa 4.*


Nas Tabelas 5 e 6, nenhuma extensão de PPI ou de PPI × volume atende ao critério incremental. A Figura 37 mostra que as estimativas diárias têm amplitude menor que o observado; as Figuras 38 e 39 separam o caminho descritivo da exposição do ganho de previsão, que permanece abaixo do limiar. O detalhe por bloco, as associações no treino e o diagnóstico de incremento ficam no caderno auxiliar `notebooks/etapa4_experimentos.ipynb`. O ensaio não promove um modelo de exposição como preditor validado.


# **Discussão e Conclusão**

A hipótese de que o desvio de paridade ponderado pelo volume acompanha os alvos não encontra o mesmo resultado em todos eles. Uma queda do RMSE em alguns blocos de datas não basta quando a maioria dos períodos não melhora ou quando o MAE aumenta, porque o critério pedia as três condições ao mesmo tempo, como mostram a Figura 29 e a Figura 32. No diesel ANP, a extensão que separa gasolina e diesel (`y_fuels`) reduz o RMSE mediano em 16,7%, mas o RMSE é menor em apenas 1 de 4 blocos: a mediana cai porque um bloco de 2026 concentra o ganho, e os outros três não melhoram. A exposição Petrobras contemporânea (`y_pbr0`) usa um único regressor, o PPI da Petrobras vezes o volume na mesma semana, e o RMSE é menor em 3 de 4 blocos contra a história só do alvo; as duas especificações não se confundem, como mostram a Figura 30 na trajetória de `y_pbr0` e a Figura 32 na comparação por bloco. O ensaio do IPCA usa a variação do preço ANP de produtores e importadores da gasolina e, por isso, não constitui evidência sobre o produto desvio × volume, como mostra a Figura 31.

Os controles sazonais, a defasagem 12 do próprio alvo, explicam parte importante da demanda e da mobilidade, como mostra a Figura 33. No varejo, a exposição reduz o RMSE mediano sobre esse controle sazonal, mas o RMSE é menor em só dois de quatro blocos, como mostra a Figura 34. Isso aproxima a análise dos Objetivos de Desenvolvimento Sustentável (ODS) 8, trabalho decente e crescimento econômico, e 11, cidades e comunidades sustentáveis, sem atribuir ao preço do combustível uma causa que o desenho não isola. Para o ODS 16, paz, justiça e instituições eficazes, o produto inclui código, hash efetivo da base, revisão de proveniência e critérios explícitos de comparação. Consistência interna do SQLite e reprodução da dummy não equivalem a auditoria independente das fontes.

Na companhia, a receita permanece com persistência na escolha feita em 2019–2021. No lucro, SARIMA(0,1,0)×(1,0,0)₄ com história completa reduz o RMSE agrupado em 0,70% frente à persistência do 1º trimestre de 2022 ao 1º trimestre de 2026, abaixo do limiar de 5%. Na margem, a configuração escolhida é SARIMA(0,0,0)×(0,0,0)₄ com constante, um nível médio da série; o nome SARIMA não implica, neste caso, uma dinâmica sazonal, e a persistência tem RMSE agrupado menor na avaliação (3,5233 contra 4,5583). O ranking depois de 2021 permanece descritivo e não escolhe outro método, precisamente porque a trava impede que a avaliação posterior vire uma nova seleção.

A correção do resíduo soma um termo de exposição à previsão-base já escolhida. Com exposição de diesel do 1º trimestre de 2024 ao 1º trimestre de 2026, o RMSE da receita sobe de 7.281,22 para 12.075,24 R＄ milhões, enquanto o do lucro cai de 4.245,58 para 2.694,51 e o RMSE da margem, de 4,56 para 4,14 pontos percentuais. A Figura 35 mostra o lucro em R＄ milhões e o nível da margem em %, com a avaliação principal em 1T24–1T26 (nove origens) e o 2T26 separado. A comparação entre 1T24–1T26 (nove origens) e 1T24–4T25 (oito origens) retira somente o 1º trimestre de 2026: as três correções aumentam o RMSE nesse recorte (receita 4,9%, lucro 9,6%, margem 87,4%), de modo que o ganho de lucro e margem nessa janela depende da inclusão do primeiro trimestre de 2026, como mostra a Figura 36. O 2º trimestre de 2026 entra à parte, mistura ganhos e perdas e não estabelece estabilidade. As subvenções da nota 4 do Informe Trimestral (ITR) limitam a interpretação; o estudo não atribui toda diferença a elas. O horizonte trimestral curto e a exposição construída com volume contemporâneo, o volume da mesma data que se prevê, mantêm esse resultado no campo descritivo.

O diferencial de retorno da PETR4 frente ao BOVA11 acrescenta um teste da hipótese do projeto sobre um alvo relativo ao mercado, distinto dos ensaios de nível de preço e de `dlog`. Nas Tabelas 5 e 6 e nas Figuras 37 a 39, o PPI em nível nas defasagens de 1 a 5 pregões e a exposição acumulada PPI × volume, com reset trimestral, não atendem ao critério incremental nos respectivos recortes. Os fechamentos COTAHIST sem ajuste por proventos excluem as distribuições do cálculo do retorno, e a subtração do BOVA11 usa peso unitário, sem estimar beta. A exposição trimestral usa o volume mensal realizado da ANP e permanece descritiva. A escolha entre especificações nos mesmos blocos de avaliação é seleção exploratória; não houve auditoria de horário de divulgação nem confirmação em amostra independente após essa escolha, de modo que o ensaio não estabelece viabilidade ex ante, causalidade nem estratégia negociável.

As 22 semanas de gasolina revistas ficam no treino da Etapa 3, mas algumas alcançam os blocos de validação da Etapa 4. Sua proveniência permanece aberta: o changelog não explica o acréscimo. A igualdade da dummy nos dois SQLite não demonstra ausência de efeito nos modelos ajustados, porque a dummy não usa os mesmos regressores. O RMSE de `dlog`, a variação logarítmica do preço, mede o erro dessa variação, não o acerto de direção (subida ou descida). `dqty_mn` mede a variação da quantidade de ações negociadas (`total_quantity`), não o volume financeiro em reais.

A viabilidade ex ante, prever só com o calendário de divulgação e a informação que já teria estado disponível na origem, permanece **não executada**. Também não houve confirmação em uma amostra independente após a seleção. Os ajustes SARIMAX (ARIMA sazonal com regressores), Prophet, LSTM (rede recorrente de memória longa) e TimesFM da Etapa 3 permanecem históricos; a grade ARIMA/SARIMA e as suavizações trimestrais não os substituem. O vídeo de apresentação ainda exige produção e publicação pelo grupo.


# **Apresentação**

Link do vídeo (apresentação com slides).


[https://youtu.be/7_of5GmiZ-Q](https://youtu.be/7_of5GmiZ-Q)


# **Referências**

ASSOCIAÇÃO BRASILEIRA DE CONCESSIONÁRIAS DE RODOVIAS. *Índice ABCR*. São Paulo: ABCR, 2026. Disponível em: https://melhoresrodovias.org.br/. Acesso em: 19 ago. 2026.

ASSOCIAÇÃO BRASILEIRA DOS IMPORTADORES DE COMBUSTÍVEIS. *PPI*. Rio de Janeiro: ABICOM, 2026. Disponível em: https://abicom.com.br/categoria/ppi/. Acesso em: 19 ago. 2026.

AGÊNCIA NACIONAL DO PETRÓLEO, GÁS NATURAL E BIOCOMBUSTÍVEIS. *Preços de produtores e importadores de derivados de petróleo e biodiesel*. Brasília: ANP, 2026. Disponível em: https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/precos-de-produtores-e-importadores-de-derivados-de-petroleo-e-biodiesel. Acesso em: 27 set. 2026.

AGÊNCIA NACIONAL DO PETRÓLEO, GÁS NATURAL E BIOCOMBUSTÍVEIS. *Vendas de combustíveis (m³)*. Brasília: ANP, 2026. Disponível em: https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-estatisticos. Acesso em: 19 ago. 2026.

BANCO CENTRAL DO BRASIL. *Sistema Gerenciador de Séries Temporais (SGS)*. Brasília: BCB, 2026. Disponível em: https://www3.bcb.gov.br/sgspub/. Acesso em: 19 ago. 2026.

B3 S.A. – BRASIL, BOLSA, BALCÃO. *Cotações históricas*. São Paulo: B3, 2026. Disponível em: https://www.b3.com.br/pt_br/market-data-e-indices/servicos-de-dados/market-data/historico/mercado-a-vista/cotacoes-historicas/. Acesso em: 19 ago. 2026.

DAS, A.; KONG, W.; SEN, R.; ZHOU, Y. A decoder-only foundation model for time-series forecasting. In: INTERNATIONAL CONFERENCE ON MACHINE LEARNING, 41., 2024, Viena. *Proceedings* [...]. [S. l.]: PMLR, 2024. v. 235. Disponível em: https://arxiv.org/abs/2310.10688. Acesso em: 25 set. 2026.

GOOGLE RESEARCH. *TimesFM*: time series foundation model. [S. l.]: GitHub, 2026. Disponível em: https://github.com/google-research/timesfm. Acesso em: 25 set. 2026.

HOCHREITER, S.; SCHMIDHUBER, J. Long short-term memory. *Neural Computation*, v. 9, n. 8, p. 1735-1780, 1997. DOI: https://doi.org/10.1162/neco.1997.9.8.1735.

HYNDMAN, R. J.; ATHANASOPOULOS, G. *Forecasting*: principles and practice. 3. ed. Melbourne: OTexts, 2021. Disponível em: https://otexts.com/fpp3/. Acesso em: 24 ago. 2026.

INSTITUTO BRASILEIRO DE GEOGRAFIA E ESTATÍSTICA. *SIDRA*: Sistema IBGE de Recuperação Automática. Rio de Janeiro: IBGE, 2026. Disponível em: https://sidra.ibge.gov.br/. Acesso em: 19 ago. 2026.

LOPES, A. O.; VIEGAS, T. O. C.; CONTE FILHO, C. G.; CARVALHO, V. S. Fatores influenciando preços de ações Petrobras (PETR4) entre 2009 e 2020. *Research, Society and Development*, v. 10, n. 7, 2021. Disponível em: https://rsdjournal.org/index.php/rsd/article/view/16294. Acesso em: 25 set. 2026.

ORGANIZAÇÃO DAS NAÇÕES UNIDAS. *Sustainable Development Goals*. Nova York: ONU, 2024. Disponível em: https://www.undp.org/sustainable-development-goals. Acesso em: 24 ago. 2026.

PETROBRAS. *Central de resultados*. Rio de Janeiro: Petrobras, 2026. Disponível em: https://www.investidorpetrobras.com.br/resultados-e-comunicados/central-de-resultados/. Acesso em: 19 ago. 2026.

PETROBRAS. *Tabelas de preços de venda às distribuidoras*. Rio de Janeiro: Petrobras, 2026. Disponível em: https://precos.petrobras.com.br. Acesso em: 19 ago. 2026.

PUENTES MÁRQUEZ, J.; OLIVEIRA RIBEIRO, C. de; RUELAS SANTOYO, E.; FIGUEROA FERNÁNDEZ, V. Ethanol fuel demand forecasting in Brazil using a LSTM recurrent neural network approach. *IEEE Latin America Transactions*, v. 19, n. 4, p. 551-558, 2021. Disponível em: https://latamt.ieeer9.org/index.php/transactions/article/view/3713. Acesso em: 25 set. 2026.

SERRANO, A. L. M. et al. Forecasting ethanol and gasoline consumption in Brazil: advanced temporal models for sustainable energy management. *Energies*, v. 18, n. 6, 1501, 2025. DOI: https://doi.org/10.3390/en18061501.

SILVA, L. A.; PERES, S. M.; BOSCARIOLI, C. *Introdução à mineração de dados com aplicações em R*. 1. ed. São Paulo: LTC, 2016.

TAYLOR, S. J.; LETHAM, B. Forecasting at scale. *The American Statistician*, v. 72, n. 1, p. 37-45, 2018. DOI: https://doi.org/10.1080/00031305.2017.1380080.